## Table of Contents

* [Part 1: Information Extraction](#Part-1:-Information-Extraction)
    * [Step 1: Load Source Datasets](#Step-1:-Load-Source-Datasets)
    * [Step 2: Create Schema for Information Extraction](#Step-2:-Create-Schema-for-Information-Extraction)
    * [Step 3: Run LLM-Based Extraction](#Step-3:-Run-LLM-Based-Extraction-(few-shot)-using-Schema)
* [Part 2: Data Loading and Profiling](#Part-2:-Data-Loading-and-Profiling)
    * [Step 1: Loading the data](#Step-1:-Loading-the-data)
    * [Step 2: Profiling](#Step-2:-Profiling)
    * [Step 3: Dropping Sparse columns](#Step-3:-Dropping-Sparse-columns)
* [Part 3: Normalization](#part-3-normalization)
    * [Step 1: Normalizing Brands](#step-1-normalizing-brands)
    * [Step 2: Normalizing Weight](#step-2-normalizing-weight)
    * [Step 3: Normalizing Storage](#step-3-normalizing-storage)
    * [Step 4: Normalizing Dimensions](#step-4-normalizing-dimensions)
    * [Step 5: Normalizing VRAM](#step-5-normalizing-vram)
    * [Step 6: Normalizing chipset_name](#step-6-normalizing-chipset_name)
    * [Step 7: Normalizing bus_type](#step-7-normalizing-bus_type)
    * [Step 8: Price Normalization](#step-8-price-normalization)

## Part 1: Information Extraction

In [ ]:
from pathlib import Path
import pandas as pd
import json
from PyDI.informationextraction import LLMExtractor
from PyDI.schemamatching import LLMBasedSchemaMatcher, SchemaTranslator
from PyDI.normalization import load_normalization_spec
from langchain_openai import ChatOpenAI

from __future__ import annotations
import os
from collections import Counter
from typing import Any

# Paths relative to this notebook
NOTEBOOK_DIR = Path(".").resolve()
INPUT_DIR = NOTEBOOK_DIR / "input"
OUTPUT_DIR = NOTEBOOK_DIR / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

### Step 1: Load Source Datasets

In [ ]:
from PyDI.io import load_json

products_1 = load_json(INPUT_DIR / "data" / "products_1.json")
products_1.attrs["dataset_name"] = "products_1"

products_2 = load_json(INPUT_DIR / "data" / "products_2.json")
products_2.attrs["dataset_name"] = "products_2"

products_3 = load_json(INPUT_DIR / "data" / "products_3.json")
products_3.attrs["dataset_name"] = "products_3"

products_4 = load_json(INPUT_DIR / "data" / "products_4.json")
products_4.attrs["dataset_name"] = "products_4"

products_1.head()

### Step 2: Create Schema for Information Extraction

In [ ]:
from typing import Optional
from pydantic import BaseModel, Field

class ProductSchema(BaseModel):

    # ── Core ──────────────────────────────────────────────────────────────────
    brand: Optional[str] = Field(default=None,
        description="Manufacturer brand name. E.g. 'Gigabyte', 'WD', 'Samsung', 'Corsair'.")
    model: Optional[str] = Field(default=None,
        description="Marketing model name, excluding brand. E.g. 'RTX 3080 Gaming OC', 'WD Blue', 'Force MP510'.")
    model_number: Optional[str] = Field(default=None,
        description="Exact SKU or part number as printed on the product or listing. E.g. 'GV-N3080GAMINGOC-10GD', 'WD60EZAZ'.")
    product_type: Optional[str] = Field(default=None,
        description="Product category. E.g.: GPU, SSD, HDD, RAM, USB_STICK.")
    chipset_name: Optional[str] = Field(default=None,
        description="Full chipset or processor name. E.g. 'GeForce RTX 3080', 'Radeon RX 7900 XTX'.")
    chipset_series: Optional[str] = Field(default=None,
        description="GPU architecture or chipset generation family. E.g. 'Ampere', 'RDNA 3', 'Ada Lovelace'.")

    # ── Capacity ──────────────────────────────────────────────────────────────
    vram: Optional[float] = Field(default=None,
        description="GPU video memory size in GB. GPUs only. E.g. 10.0 for 10GB VRAM.")
    storage_size: Optional[float] = Field(default=None,
        description="Storage capacity of SSDs, HDDs, and USB sticks only.")

    # ── Performance ───────────────────────────────────────────────────────────
    base_core_clock: Optional[float] = Field(default=None,
        description="GPU base/reference core clock speed. E.g. 1440.0.")
    boost_core_clock: Optional[float] = Field(default=None,
        description="GPU maximum boost core clock speed. E.g. 1800.0.")
    graphics_interface_speed: Optional[float] = Field(default=None,
        description="PCIe interface transfer speed. E.g. 16.0 for PCIe 4.0, 32.0 for PCIe 5.0. Not the memory bus width.")
    memory_clock: Optional[float] = Field(default=None,
        description="Effective memory clock speed. E.g. For GDDR6X '19 Gbps'. For storage, this is typically null.")
    read_speed: Optional[float] = Field(default=None,
        description="Sequential read speed. SSDs and HDDs only. E.g. 3500.0.")
    write_speed: Optional[float] = Field(default=None,
        description="Sequential write speed. SSDs and HDDs only. E.g. 3000.0.")
    random_4k_read_iops: Optional[float] = Field(default=None,
        description="Random 4K read performance in IOPS. SSDs only. E.g. 500000.0.")
    random_4k_write_iops: Optional[float] = Field(default=None,
        description="Random 4K write performance in IOPS. SSDs only. E.g. 480000.0.")

    # ── Connection ────────────────────────────────────────────────────────────
    bus_type: Optional[str] = Field(default=None,
        description="Host interface bus type and version. E.g. 'PCIe 4.0 x16', 'PCIe 3.0 x4', 'USB 3.2 Gen 2', 'SATA III'.")
    memory_interface_width: Optional[int] = Field(default=None,
        description="GPU memory bus width in bits. E.g. 128, 192, 256, 320, 384. Only for GPUs; sometimes labelled 'Memory Interface' on spec sheets.")
    interface_type: Optional[str] = Field(default=None,
        description="Logical or physical connector interface. E.g. 'NVMe', 'SATA', 'Type-C', 'M.2'. Distinct from bus_type.")
    protocol: Optional[str] = Field(default=None,
        description="Communication protocol with version. E.g. 'NVMe 1.4', 'NVMe 2.0', 'AHCI'. Usually for SSDs.")

    # ── GPU-Specific ──────────────────────────────────────────────────────────
    max_resolution: Optional[str] = Field(default=None,
        description="Maximum display output resolution supported. E.g. '7680x4320', '4K'.")
    max_monitor_support: Optional[int] = Field(default=None,
        description="Maximum number of simultaneous displays supported. E.g. 4.")
    video_out_interface: Optional[str] = Field(default=None,
        description="Display output connector types and versions, comma-separated. E.g. 'HDMI 2.1, DP 1.4a, USB-C'.")
    core_shader_count: Optional[int] = Field(default=None,
        description="Total number of shader processors / CUDA cores / stream processors. E.g. 8704.")
    ray_tracing_units: Optional[int] = Field(default=None,
        description="Number of dedicated ray tracing cores or units. E.g. 68 for RTX 3080.")
    ai_cores: Optional[int] = Field(default=None,
        description="Number of dedicated AI/tensor cores. E.g. 272 for RTX 3080.")
    adaptive_sync: Optional[str] = Field(default=None,
        description="Adaptive sync technology supported. E.g. 'AMD FreeSync Premium', 'NVIDIA G-Sync Compatible', 'G-Sync'.")
    upscaling_technology: Optional[str] = Field(default=None,
        description="AI or spatial upscaling tech with version if available. E.g. 'DLSS 3.5', 'FSR 3', 'XeSS'.")
    overclocked: Optional[bool] = Field(default=None,
        description="True if the card ships with factory overclock above reference spec (look for OC, XT, or similar in model name). False if reference/stock clocks.")
    directx_version: Optional[str] = Field(default=None,
        description="Highest DirectX version supported. E.g. '12 Ultimate', '12'.")
    opengl_version: Optional[str] = Field(default=None,
        description="Highest OpenGL version supported. E.g. '4.6'.")
    shader_model_version: Optional[str] = Field(default=None,
        description="Highest shader model version supported. E.g. '6.7'.")

    # ── Physical Dimensions & Features ───────────────────────────────────────
    width: Optional[float] = Field(default=None,
        description="Card or device width.")
    length: Optional[float] = Field(default=None,
        description="Card or device length/depth.")
    height: Optional[float] = Field(default=None,
        description="Card or device height. For GPUs this is slot thickness.")
    weight: Optional[float] = Field(default=None,
        description="Device weight.")
    fan_count: Optional[int] = Field(default=None,
        description="Number of cooling fans on the device. E.g. 2 or 3 for GPUs.")
    cooling_type: Optional[str] = Field(default=None,
        description="Cooling solution type. E.g. 'active air', 'active liquid', 'passive'. Water/liquid cooling is considered active.")
    storage_connection_type: Optional[str] = Field(default=None,
        description="Physical connector type for storage devices. E.g. 'M.2', '2.5-inch SATA', '3.5-inch SATA', 'USB-A'.")

    # ── Reliability & Power ───────────────────────────────────────────────────
    power_draw_w: Optional[float] = Field(default=None,
        description="Typical or maximum power consumption. E.g. 320.0.")
    power_adapter: Optional[str] = Field(default=None,
        description="Required power connector type. E.g. '2x 8-pin', '16-pin 600W', '1x 6-pin'.")
    TBW_rating: Optional[float] = Field(default=None,
        description="Total Bytes Written endurance rating. SSDs only. E.g. 600.0.")
    MTBF_hours: Optional[float] = Field(default=None,
        description="Mean Time Between Failures. E.g. 1500000.0.")
    operating_temperature: Optional[str] = Field(default=None,
        description="Operating temperature range as a string. E.g. '0°C to 70°C'.")

    # ── Technical Specs ───────────────────────────────────────────────────────
    L1_cache: Optional[float] = Field(default=None,
        description="L1 cache size.")
    L2_cache: Optional[float] = Field(default=None,
        description="L2 cache size. For GPUs this may be listed as 'L2 Cache'. E.g. 5120.0 for RTX 3080.")
    L3_cache: Optional[float] = Field(default=None,
        description="L3 cache size. Primarily for CPUs; rare on GPUs.")
    manufacturing_process: Optional[str] = Field(default=None,
        description="Fabrication node as a string. E.g. '8nm', '5nm', '7nm'. Keep the unit in the string.")
    memory_type: Optional[str] = Field(default=None,
        description="Memory technology type. E.g. 'GDDR6X', 'GDDR6', 'DDR5', 'LPDDR4X', 'HBM2e'.")
    encryption_type: Optional[str] = Field(default=None,
        description="Hardware encryption standard supported. E.g. 'AES-256', 'TCG Opal 2.0'.")
    controller: Optional[str] = Field(default=None,
        description="Storage controller chip model. E.g. 'Phison E18', 'Samsung Elpis'. SSDs only.")
    dual_bios: Optional[bool] = Field(default=None,
        description="True if the GPU has a dual BIOS / dual firmware switch.")
    storage_technology: Optional[str] = Field(default=None,
        description="Recording or NAND technology. E.g. 'CMR', 'SMR' for HDDs; 'TLC', 'QLC', 'MLC', '3D NAND' for SSDs.")

    # ── Design & Aesthetics ───────────────────────────────────────────────────
    color: Optional[str] = Field(default=None,
        description="Primary color or finish of the product. E.g. 'Black', 'White', 'Silver'.")
    form_factor: Optional[str] = Field(default=None,
        description="Physical form factor standard. E.g. 'M.2 2280', '3.5-inch', '2.5-inch', 'ATX'.")
    blocked_slots: Optional[int] = Field(default=None,
        description="Number of PCIe expansion slots occupied by the GPU cooler. E.g. 2 or 3.")
    case_material: Optional[str] = Field(default=None,
        description="Primary housing material. E.g. 'Aluminum', 'Plastic', 'Steel'.")
    delivery_scope: Optional[str] = Field(default=None,
        description="Retail packaging tier. E.g. 'OEM' (no box/accessories), 'Retail' (full box), 'Bulk'.")

**quick check on individual source files.**

In [ ]:
#check how many entities we have for each dataset and get totals
datasets = [products_1, products_2, products_3, products_4]
total_entities = 0
for i, dataset in enumerate(datasets, start=1):
    num_entities = len(dataset)
    total_entities += num_entities

    print(f"Dataset {i} ({dataset.attrs['dataset_name']}): {num_entities} entities")
print(f"Total entities across all datasets: {total_entities}")

### Step 3: Run LLM-Based Extraction (few shot) using Schema

#### Defining the prompt

In [ ]:
import tiktoken
import json
from dotenv import load_dotenv
load_dotenv()

enc = tiktoken.get_encoding("o200k_base")

# ── Token counter helper ──────────────────────────────────────────────────────
def count_tokens(text: str) -> int:
    return len(enc.encode(str(text)))

# ── Load few-shot examples ────────────────────────────────────────────────────
with open(INPUT_DIR / "few_shot_examples.json", "r") as f:
    few_shot_examples = json.load(f)

# ── Raw inputs paired with their ground truth extractions ────────────────────
few_shot_inputs = [
    "SanDisk Cruzer Blade 128GB USB 2.0 Pen Drive. Description: ENJOY AMPLE CAPACITIES AND MAXIMUM PORTABILITY. With its stylish, compact design and generous capacity, the Cruzer Blade USB Flash Drive makes it easy to back up, transfer, and share your files.",
    "Corsair Force MP510 M.2 SSD - 960GB. Description: Solid State Drive, 960 GB, intern, M.2 2280, PCI Express 3.0 x4 (NVMe), 256-bit AES",
    "Gigabyte NVIDIA GeForce RTX 3080 Gaming OC 10GB Ampere Graphics Card. Description: CUDA Cores: 8704, Boost Clock: 1800MHz, GDDR6X, 320-bit Memory, Memory Clock: 19.0 Gbps, Ray Tracing, 3x Display Port, 2x HDMI, 3 Yr Warranty.",
    "WD Blue 6TB 3.5\" SATA 3 HDD/Hard Drive. Description: 6TB WD Blue WD60EZAZ, 3.5\" HDD, SATA III - 6Gb/s, 5400rpm, 256MB Cache, OEM NCQ",
]

# ── Build few-shot block ──────────────────────────────────────────────────────
few_shot_block = "\n\n".join(
    f"<example>\n<input>{inp}</input>\n<output>{json.dumps(out, indent=2)}</output>\n</example>"
    for inp, out in zip(few_shot_inputs, few_shot_examples)
)

# ── System prompt ─────────────────────────────────────────────────────────────
SYSTEM_PROMPT = f"""You are a world-class algorithm for extracting information in structured formats. Extract the product attribute values from the provided text into a JSON object adhering to the provided schema.

Classification: Identify if the product is a GPU, SSD, HDD, or USB_STICK.
Precision: Extract values exactly as they appear in the text where possible.
Handling Missing Data: If an attribute is not present in the text, you must return null as per the schema. DO NOT make up information.

Here are examples of ideal extractions:

{few_shot_block}"""

system_prompt_tokens = count_tokens(SYSTEM_PROMPT)
print(f"System prompt tokens (constant per row): {system_prompt_tokens:,}")



#### LLM Extraction

In [ ]:
# ── Combine title + description ───────────────────────────────────────────────
for dataset in [products_1, products_2, products_3, products_4]:
    dataset["title_description"] = (
        dataset[["title", "description"]]
        .fillna("").astype(str)
        .agg(". Description: ".join, axis=1)
    )
# ── Extractor ─────────────────────────────────────────────────────────────────
extractor = LLMExtractor(
    chat_model=ChatOpenAI(model="gpt-5.2", temperature=0),
    source_column="title_description",
    schema=ProductSchema,
    system_prompt=SYSTEM_PROMPT,  # improved prompt with few-shot examples
)
# ── Extract ───────────────────────────────────────────────────────────────────
extracted_products_1 = extractor.extract(products_1)
extracted_products_2 = extractor.extract(products_2)
extracted_products_3 = extractor.extract(products_3)
extracted_products_4 = extractor.extract(products_4)
# ── Token counts ──────────────────────────────────────────────────────────────
schema_fields = list(ProductSchema.model_fields.keys())
for name, raw, extracted in [
    ("products_1", products_1, extracted_products_1),
    ("products_2", products_2, extracted_products_2),
    ("products_3", products_3, extracted_products_3),
    ("products_4", products_4, extracted_products_4),
]:
    input_tokens  = raw["title_description"].apply(lambda t: system_prompt_tokens + count_tokens(t)).sum()
    output_tokens = extracted[schema_fields].apply(lambda row: count_tokens(row.to_json()), axis=1).sum()
    print(f"{name} — input: {input_tokens:,} | output: {output_tokens:,} | total: {input_tokens + output_tokens:,}")
# ── Save ──────────────────────────────────────────────────────────────────────
extracted_products_1.to_json(OUTPUT_DIR / "products_1_extracted.json", orient="records", indent=2)
extracted_products_2.to_json(OUTPUT_DIR / "products_2_extracted.json", orient="records", indent=2)
extracted_products_3.to_json(OUTPUT_DIR / "products_3_extracted.json", orient="records", indent=2)
extracted_products_4.to_json(OUTPUT_DIR / "products_4_extracted.json", orient="records", indent=2)


In [ ]:
# Save to json
extracted_products_1.to_json(OUTPUT_DIR / "informationextraction" / "extracted_products_1.json", orient="records", indent=2)

In [ ]:
# Run extraction for other datasets
extracted_products_2 = extractor.extract(products_2)
extracted_products_3 = extractor.extract(products_3)
extracted_products_4 = extractor.extract(products_4)

# Save to json
extracted_products_2.to_json(OUTPUT_DIR / "informationextraction" / "extracted_products_2.json", orient="records", indent=2)
extracted_products_3.to_json(OUTPUT_DIR / "informationextraction" / "extracted_products_3.json", orient="records", indent=2)
extracted_products_4.to_json(OUTPUT_DIR / "informationextraction" / "extracted_products_4.json", orient="records", indent=2)

## Part 2: Data Loading and Profiling

### Step 1: Loading the data

In [ ]:
import os
import pandas as pd
import json
from pathlib import Path

# Paths relative to this notebook
NOTEBOOK_DIR = Path(".").resolve()
INPUT_DIR = NOTEBOOK_DIR / "input"
OUTPUT_DIR = NOTEBOOK_DIR / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

#load back the extracted data
extracted_products_1 = pd.read_json(OUTPUT_DIR / "informationextraction" / "LLM_extracted_products_data" / "products_1_extracted.json", orient="records")
extracted_products_2 = pd.read_json(OUTPUT_DIR / "informationextraction" / "LLM_extracted_products_data" / "products_2_extracted.json", orient="records")
extracted_products_3 = pd.read_json(OUTPUT_DIR / "informationextraction" / "LLM_extracted_products_data" / "products_3_extracted.json", orient="records")
extracted_products_4 = pd.read_json(OUTPUT_DIR / "informationextraction" / "LLM_extracted_products_data" / "products_4_extracted.json", orient="records")

In [ ]:
from PyDI.io import load_json

# Load extracted products dataset
products_1_extracted = load_json(
    OUTPUT_DIR / "informationextraction" / "LLM_extracted_products_data" / "products_1_extracted.json",
    name="products_1_extracted"
    )

products_2_extracted = load_json(
    OUTPUT_DIR / "informationextraction" / "LLM_extracted_products_data" / "products_2_extracted.json",
    name="products_2_extracted"
    )


products_3_extracted = load_json(
    OUTPUT_DIR / "informationextraction" / "LLM_extracted_products_data" / "products_3_extracted.json",
    name="products_3_extracted"
    )

products_4_extracted = load_json(
    OUTPUT_DIR / "informationextraction" / "LLM_extracted_products_data" / "products_4_extracted.json",
    name="products_4_extracted"
    )

# Display basic information
datasets = [products_1_extracted, products_2_extracted, products_3_extracted,products_4_extracted]
names = ["products_1_extracted", "products_2_extracted", "products_3_extracted", "products_4_extracted"]

total_records = sum(len(df) for df in datasets)
print(f"Total records across all datasets: {total_records:,}")

### Step 2: Profiling

In [ ]:
from PyDI.utils import DataProfiler

# Initialize the DataProfiler
profiler = DataProfiler()

for df, name in zip(datasets, names):
    profile = profiler.summary(df) # automatically prints some statistics and returns object containing stats

display(profile)

### Attribute Coverage Analysis  

In [ ]:
coverage = profiler.analyze_coverage(
    datasets=datasets,
    include_samples=True,
    sample_count=3  # Show 3 sample values per attribute
)
# 1. Force pandas to show all 66 rows
pd.set_option('display.max_rows', None)

# 2. Force pandas to show all 20 columns
pd.set_option('display.max_columns', None)

# 3. Prevent text inside cells from being cut off
pd.set_option('display.max_colwidth', None)

print("📊 Attribute coverage across datasets:")
display(coverage)

# Identify attributes suitable for entity matching
print("\n🔗 Attributes suitable for entity matching:")
matching_attrs = coverage[coverage['datasets_with_attribute'] >= 2]['attribute'].tolist()
print(f"Attributes available in 2+ datasets: {matching_attrs}")


In [ ]:
import pandas as pd

# 1. Basic parameters
total_records = sum(len(df) for df in datasets)
# Generic columns to keep regardless of density
protected_generics = [
    'height', 'width', 'length', 'weight', 'brand', 'model', 
    'title', 'price', 'id', 'url', 'cluster_id', 'model_number'
]

# 2. Extract numeric counts from strings like '37/812'
count_cols = [c for c in coverage.columns if c.endswith('_count')]

def get_raw_count(val):
    if pd.isna(val) or not isinstance(val, str): return 0
    try:
        return int(val.split('/')[0])
    except:
        return 0

# Calculate total non-nulls and weighted density
coverage['total_non_nulls_sum'] = coverage[count_cols].applymap(get_raw_count).sum(axis=1)
coverage['global_density_pct'] = (coverage['total_non_nulls_sum'] / total_records) * 100

# 3. Apply Decision Logic
def decide_action(row):
    if row['attribute'] in protected_generics:
        return "KEEP (Generic Exception)"
    if row['global_density_pct'] >= 15.0:
        return "KEEP (Passes 15%)"
    return "DROP (Low Density)"

coverage['decision'] = coverage.apply(decide_action, axis=1)

# 4. Show the results sorted by density
decision_table = coverage[[
    'attribute', 'total_non_nulls_sum', 'global_density_pct', 'decision'
]].sort_values(by='global_density_pct', ascending=False)

print(f"📊 Total Dataset Size: {total_records} records")
display(decision_table)

In [ ]:
#make a copy of each dataset to play with.
extracted_products_1_copy = extracted_products_1.copy()
extracted_products_2_copy = extracted_products_2.copy()
extracted_products_3_copy = extracted_products_3.copy()
extracted_products_4_copy = extracted_products_4.copy()

all_datasets_copy = [extracted_products_1_copy, extracted_products_2_copy, extracted_products_3_copy, extracted_products_4_copy]

### Step 3: Dropping Sparse columns

In [ ]:
# global_drop = [
#     'L1_cache', 'L2_cache', 'L3_cache', 'ai_cores','shader_model_version', 
#     'upscaling_technology', 'ray_tracing_units', 'dual_bios'
# ]

# df_after_dropping = [df.drop(columns=global_drop) for df in all_datasets]


global_drop = [
    #at first dropped only under 1% covearge.
    'L1_cache', 'L2_cache', 'L3_cache', 'ai_cores',
    'dual_bios', 'upscaling_technology','shader_model_version', 'ray_tracing_units',
    #now we also drop stuff under 15%avg coverage across datasets
    # excluding generic attributes like
    # weight,dimensions,etc even if they are below 15%.
    'adaptive_sync','blocked_slots','graphics_interface_speed','power_adapter','opengl_version',
    'manufacturing_process','max_monitor_support','operating_temperature', 
    'power_draw_w','case_material', 'directx_version', 'controller',
    'MTBF_hours', 'encryption_type', 'fan_count', 'cooling_type',
    'max_resolution', 'memory_clock', 'base_core_clock', 
    'random_4k_read_iops', 'random_4k_write_iops','overclocked', 
    'delivery_scope','core_shader_count', 'chipset_series',
    'protocol', 'boost_core_clock','memory_interface_width', 'video_out_interface', 'TBW_rating',
    #color is considered generic so wont drop it
    'storage_technology' #14.4%
    #write_speed not dropped as its 14.7% and read is 17% so should be together.
]

# 1. Print the stats for the dropped columns before we lose them
print("🗑️ Summary of Dropped Column Coverage:")
print("-" * 60)

for i, df in enumerate(all_datasets_copy, start=1):
    print(f"\nDataset {i} (Total Rows: {len(df)})")
    # Identify which columns from our drop list actually exist in this dataframe
    cols_to_check = [c for c in global_drop if c in df.columns]
    
    if not cols_to_check:
        print("  No matching columns found to drop.")
        continue

    for col in cols_to_check:
        count = df[col].count()  # Counts non-null values
        pct = (count / len(df)) * 100
        print(f"  - {col:22} | Fill Count: {count:4} | {pct:6.2f}% full")

# 2. Perform the actual drop
df_after_dropping = [df.drop(columns=global_drop, errors='ignore') for df in all_datasets_copy]

# 3. Final verification of remaining columns
print("\n" + "="*60)
for i, df in enumerate(df_after_dropping, start=1):
    print(f"Dataset {i} - Final column count: {len(df.columns)}")

**we also check coverage locally for each product type**

In [ ]:
#before we proceed,we drop attributes with less than 1% coverage across all datasets based on profiling results.

import pandas as pd

# Define the categories to analyze
categories = ['GPU', 'HDD', 'SSD', 'USB_STICK']
#brand_normalized_datasets

# Set display options for better visibility
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

for category in categories:
    print(f"\n" + "="*50)
    print(f"🔍 CATEGORY ANALYSIS: {category}")
    print("="*50)
    
    # 1. Filter each dataset for the specific category
    filtered_group = []
    for i, df in enumerate(df_after_dropping, start=1):
        subset = df[df['product_type'] == category]
        filtered_group.append(subset)
        print(f"Dataset {i} - {category} Count: {len(subset)}")

    # 2. Run coverage analysis on the subset
    if any(len(s) > 0 for s in filtered_group):
        category_coverage = profiler.analyze_coverage(
            datasets=filtered_group,
            include_samples=True,
            sample_count=3
        )
        
        print(f"\n📊 Attribute coverage for {category}:")
        display(category_coverage)
        
        # 3. Highlight attributes that have > 50% coverage in this category
        good_attrs = category_coverage[category_coverage['avg_coverage'] > 0.5]['attribute'].tolist()
        print(f"✅ Strong matching candidates for {category}: {good_attrs}")
    else:
        print(f"⚠️ No data found for category: {category}")

**We drop all columns under 15% coverage.**

**we save the datasets again before normalization.**

In [ ]:
#save dataseets again before normalization. 
os.makedirs(INPUT_DIR / "data_cleaning" / "informationextraction_then_dropped_columns", exist_ok=True)


df_after_dropping[0].to_json(INPUT_DIR / "data_cleaning" / "informationextraction_then_dropped_columns" / "products_1.json", orient="records", indent=2)
df_after_dropping[1].to_json(INPUT_DIR / "data_cleaning" / "informationextraction_then_dropped_columns" / "products_2.json", orient="records", indent=2)
df_after_dropping[2].to_json(INPUT_DIR / "data_cleaning" / "informationextraction_then_dropped_columns" / "products_3.json", orient="records", indent=2)
df_after_dropping[3].to_json(INPUT_DIR / "data_cleaning" / "informationextraction_then_dropped_columns" / "products_4.json", orient="records", indent=2)

In [ ]:
from PyDI.io import load_json

products_1 = load_json(INPUT_DIR / "data" / "products_1.json")
products_1.attrs["dataset_name"] = "products_1"

products_2 = load_json(INPUT_DIR / "data" / "products_2.json")
products_2.attrs["dataset_name"] = "products_2"

products_3 = load_json(INPUT_DIR / "data" / "products_3.json")
products_3.attrs["dataset_name"] = "products_3"

products_4 = load_json(INPUT_DIR / "data" / "products_4.json")
products_4.attrs["dataset_name"] = "products_4"

products_1.head()

In [ ]:
#print prod1 head with full width to see all attrs
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)    
print(products_1.head())

#save first 3 lines in a file and last 3 lines in another file to see all attrs also few examples where the title has usb and few examples where the title has ssd


#save few examples where the title has usb and few examples where the title has ssd, has hdd, has rtx
products_1[products_1['title'].str.contains('USB', case=False)].head(1).to_csv('products_1_usb_examples.csv', index=False)
products_1[products_1['title'].str.contains('SSD', case=False)].head(1).to_csv('products_1_ssd_examples.csv', index=False)
products_1[products_1['title'].str.contains('HDD', case=False)].head(1).to_csv('products_1_hdd_examples.csv', index=False)
products_1[products_1['title'].str.contains('RTX', case=False)].head(1).to_csv('products_1_rtx_examples.csv', index=False)


**Shipped sources.** The task's `input/data/dataset_<n>.json` files are the output of **Part 2** of this notebook, i.e. the values as the LLM extractor produced them, **before** the Part 3 normalization below; normalizing them is part of the task. The four combined title/description columns were removed and the rows were shuffled deterministically (seed `madi-products-extracted-2026-09-22-v1`); the ids are unchanged, and the released sources do not contain `cluster_id`.



## Part 3: Normalization

**After dropping some columns, we need to clean the data by normalizing values**

### Step 1: Normalizing Brands

In [ ]:
brand_full_name_map = {
    # Western Digital Cluster
    "wd": "Western Digital",
    "wd western digital": "Western Digital",
    "western digital": "Western Digital",
    "western digital corporation": "Western Digital",
    "wd_black": "Western Digital",
    "wd Black": "Western Digital",
    "wd_Black": "Western Digital",
    "wd_black": "Western Digital",
    "ironwolf": "Seagate", # Often listed as brand in some extractions
    "g-technology": "Western Digital",
    
    # Seagate Cluster
    "seagate technology": "Seagate",
    "maxtor/seagate": "Seagate",
    "skyhawk": "Seagate",
    "nytro": "Seagate",
    "exos": "Seagate",
    "maxtor": "Seagate",
    "barracuda": "Seagate",
    "firecuda": "Seagate",
    "maxtor": "Seagate",
    "hikvision seagate": "Seagate",
    "seagate lacie": "Seagate",
    "sg": "Seagate",

    # Asus Cluster
    "asus": "Asus",
    "asus rog": "Asus",
    "华硕": "Asus",
    "rog strix": "Asus",
    
    # Gigabyte Cluster
    "gigabyte": "Gigabyte",
    "aorus": "Gigabyte", 
    "gigabyte technology": "Gigabyte",
    
    # Kingston Cluster
    "kingston": "Kingston",
    "kingston technology": "Kingston",
    "kingston digital": "Kingston",
    "hyperx": "Kingston",
    
    # HP & HPE (Kept separate and expanded to full names)
    "hp": "Hewlett Packard",
    "hewlett packard hp": "Hewlett Packard",
    "hewlett packard": "Hewlett Packard",
    "hpe": "Hewlett Packard Enterprise",
    "hp enterprise": "Hewlett Packard Enterprise",
    "hewlett packard enterprise": "Hewlett Packard Enterprise",
    "hewlett-packard": "Hewlett Packard",
    "midline": "Hewlett Packard Enterprise",
    
    # ADATA Cluster
    "adata": "ADATA",
    "a-data": "ADATA",
    "xpg": "ADATA",
    "adata xpg": "ADATA",

    
    # Crucial/Micron
    "micron": "Crucial",
    "crucial": "Crucial",


    # Others
    "sandisk": "SanDisk",
    "sapphire technology": "Sapphire",
    "zotac gaming": "Zotac",
    "la cie": "LaCie",
    "patriot memory": "Patriot",
    "team group": "Team Group",
    "teamgroup": "Team Group",
    "team": "Team Group",
    "powercolor": "PowerColor",
    "pny": "PNY",
    "nvidia": "Nvidia",
    "geforce": "Nvidia",
    "pny technology": "PNY",
    "palit": "Palit",
    "intel®": "Intel",
    "dell equallogic": "Dell",
    "radeon": "AMD",
    "zotac gaming": "Zotac",
}

In [ ]:
def apply_comprehensive_normalization(df):
    if 'brand' in df.columns:
        # Ensure string type and clean
        df['brand'] = df['brand'].astype(str).str.lower().str.strip()
        
        # Mapping (Dictionary keys must be lowercase!)
        df['brand'] = df['brand'].replace(brand_full_name_map)
        
        # Formatting
        acronyms = ['msi', 'evga', 'pny', 'xfx', 'ssd', 'adata', 'emc', 'amd']
        
        df['brand'] = df['brand'].apply(
            lambda x: str(x).upper() if str(x).lower() in acronyms else str(x).title()
        )
        
        # Cleanup 'None' strings produced by .astype(str)
        # We catch lowercase 'none' and 'nan' because of Step 1
        df['brand'] = df['brand'].replace({'None': None, 'Nan': None, 'none': None, 'nan': None})
        
    return df

# Apply the comprehensive normalization to all datasets
extracted_products_1_brand_norm = apply_comprehensive_normalization(df_after_dropping[0])
extracted_products_2_brand_norm = apply_comprehensive_normalization(df_after_dropping[1])
extracted_products_3_brand_norm = apply_comprehensive_normalization(df_after_dropping[2])
extracted_products_4_brand_norm = apply_comprehensive_normalization(df_after_dropping[3])

In [ ]:
brand_normalized_datasets = {
    "Extracted_Products_1_brand_Normalized": extracted_products_1_brand_norm,
    "Extracted_Products_2_brand_Normalized": extracted_products_2_brand_norm,
    "Extracted_Products_3_brand_Normalized": extracted_products_3_brand_norm,
    "Extracted_Products_4_brand_Normalized": extracted_products_4_brand_norm
}
brand_normalized_datasets_list = list(brand_normalized_datasets.values())

def check_all_values(brand_normalized_datasets, column):
    """Prints EVERY unique value for a column across all datasets."""
    print(f"🔍 FULL Value Distribution for column: '{column}'")
    print("-" * 60)
    
    for name, df in brand_normalized_datasets.items():
        print(f"--- Dataset: {name} ---")
        if column in df.columns:
            # Getting all counts without .head()
            counts = df[column].value_counts(dropna=False)
            
            print(f"Total Unique Values (including NaNs): {len(counts)}")
            print(f"{'Value':<35} | Count")
            print("-" * 45)
            
            for val, count in counts.items():
                # Using str(val) handles NaNs and numeric types gracefully
                print(f"  {str(val)[:30]:<35} | {count}")
        else:
            print(f"  Attribute '{column}' not found in this dataset.")
        print("\n")

check_all_values(brand_normalized_datasets, 'brand')
# check_top_values(datasets, 'memory_gb')

**we check certain attributes for normalization**

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

def plot_mini_distributions(datasets, attributes):
    """
    Plots a row of 4 mini-distributions for each attribute.
    """
    for attr in attributes:
        # Check if column exists in at least one dataset
        if not any(attr in df.columns for df in datasets):
            continue
            
        fig, axes = plt.subplots(1, 4, figsize=(20, 4), sharey=False)
        fig.suptitle(f"🔍 Attribute: {attr}", fontsize=16, fontweight='bold', y=1.05)
        
        for i, df in enumerate(datasets):
            ax = axes[i]
            if attr not in df.columns or df[attr].dropna().empty:
                ax.text(0.5, 0.5, "No Data", ha='center', va='center')
                ax.set_title(f"Dataset {i+1}")
                continue
            
            data = df[attr].dropna()
            
            # Determine if numeric or categorical
            if pd.api.types.is_numeric_dtype(data):
                sns.histplot(data, kde=True, ax=ax, color='skyblue')
                ax.set_xlabel("Value")
            else:
                # Show top 7 categories to prevent overcrowding
                top_cats = data.value_counts().head(7)
                sns.barplot(x=top_cats.values, y=top_cats.index, ax=ax, palette='viridis',hue=0.6)
                ax.set_xlabel("Count")
            
            ax.set_title(f"Dataset {i+1} (n={len(data)})")
        
        plt.tight_layout()
        plt.show()

# List of attributes to check
check_list = [     
    'price', 'priceCurrency', 'chipset_name', 'vram', 'storage_size',
    'read_speed', 'write_speed', 'bus_type', 'interface_type',
    'width', 'length', 'height', 'weight', 
    'storage_connection_type', 'memory_type', 'color', 'form_factor'
]

# Run the visualization
plot_mini_distributions(brand_normalized_datasets_list, check_list)

In [ ]:
attributes_to_check = [
    'height', 'width', 'length', 'weight',
    'storage_size', 'write_speed', 'read_speed'
]

for attr in attributes_to_check:
    print(f"\n🔍 Attribute: {attr}")
    print("-" * 70)
    
    for name, df in brand_normalized_datasets.items():
        if attr not in df.columns:
            print(f"   {name}: Not present")
            continue
            
        col_data = df[attr].dropna()
        if col_data.empty:
            print(f"   {name}: Empty")
            continue
                
            # Get samples (unique values to see formatting)
        samples = col_data.astype(str).unique()[:5]
        
        if pd.api.types.is_numeric_dtype(col_data):
            stats = col_data.describe()
            print(f"   {name} | Numeric | Min: {stats['min']:8.2f} | Max: {stats['max']:8.2f} | Mean: {stats['mean']:8.2f}")
        else:
            print(f"   {name} | String  | Unique Count: {col_data.nunique()}")
        
        print(f"             Samples: {samples}")

In [ ]:
#check rows where weight is above 900
ds_list = list(brand_normalized_datasets.values())

df_filtered = ds_list[0][ds_list[1]['weight'] < 1] #to check if grams

#display and print count of such rows
print(f"Rows with weight < 1: {len(df_filtered)}")
display(df_filtered[['title_description', 'weight']])

#save all rows not exceeding 1 weight from all datasets to a json for review
# filtered_dfs = []
# for df in df_after_dropping:
#     filtered = df[df['weight'] < 1]
#     filtered_dfs.append(filtered)
# # Concatenate all filtered dataframes
# final_filtered = pd.concat(filtered_dfs, ignore_index=True) 
# final_filtered.to_json(OUTPUT_DIR / "informationextraction_then_dropped_columns" / "products_weight_below_1.json", orient="records", indent=2)  



In [ ]:
import pandas as pd

WEIGHT_RANGES = {
    "USB_STICK": (3, 60),     # raised to include heavier custom drives
    "SSD": (5, 300),
    "HDD": (100, 2000),       # raised to include large desktop HDDs
    "GPU": (150, 2000)        # lowered to include low-profile cards
}

KG_THRESHOLD = {
    "USB_STICK": 60,
    "SSD": 300,
    "HDD": 2000,
    "GPU": 2000
}

OZ_RANGES = {
    "USB_STICK": (0.1, 2),
    "SSD":       (0.5, 5),
    "HDD":       (10, 40),    # raised to include heavier enterprise HDDs in oz
    "GPU":       (5, 75),     # lowered to include light low-profile cards in oz
}
OZ_TO_G = 28.3495

def convert_weight_to_grams(weight, product_type):
    weight = float(weight)
    gram_low, gram_high = WEIGHT_RANGES.get(product_type, (0, float("inf")))

    # 1. Already valid grams
    if gram_low <= weight <= gram_high:
        return weight

    # 2. Likely kg — try any value under 10 first
    if 0 < weight < 10:
        converted = weight * 1000
        if gram_low <= converted <= gram_high:
            return converted

    # 3. Likely oz
    oz_low, oz_high = OZ_RANGES.get(product_type, (0, 0))
    if oz_low <= weight <= oz_high:
        converted = weight * OZ_TO_G
        if gram_low <= converted <= gram_high:
            return converted

    # 4. Nothing worked
    return weight


def check_range(weight, product_type):
    if product_type not in WEIGHT_RANGES:
        return None
    low, high = WEIGHT_RANGES[product_type]
    return low <= weight <= high


def dry_run_weight_check(df):
    df = df.copy()

    has_weight = df["weight"].notna()
    known_type = df["product_type"].isin(WEIGHT_RANGES.keys())
    mask = has_weight & known_type

    print(f"Total rows:                    {len(df)}")
    print(f"Rows with weight + known type: {mask.sum()}")
    print(f"Rows skipped (null/unknown):   {(~mask).sum()}\n")

    df_check = df[mask].copy()

    # Convert
    df_check["weight_converted"] = df_check.apply(
        lambda row: convert_weight_to_grams(row["weight"], row["product_type"]), axis=1
    )

    # Validate
    df_check["valid_weight"] = df_check.apply(
        lambda row: check_range(row["weight_converted"], row["product_type"]), axis=1
    )

    valid   = df_check["valid_weight"].sum()
    invalid = (~df_check["valid_weight"]).sum()

    print(f"Valid weights after conversion:   {valid} ({valid/len(df_check):.2%})")
    print(f"Invalid weights after conversion: {invalid} ({invalid/len(df_check):.2%})")

    if invalid > 0:
        print("\n⚠️ Invalid rows (manual inspection needed):")
        display(df_check[~df_check["valid_weight"]][
            ["brand", "product_type", "weight", "weight_converted", "title"]
        ].head(10))

    df.loc[mask, "weight_converted"] = df_check["weight_converted"]
    df.loc[mask, "valid_weight"]     = df_check["valid_weight"]

    


dry_run_weight_check(brand_normalized_datasets_list[3])

### Step 2: Normalizing Weight

In [ ]:
def apply_weight_normalization(df):
    """
    Applies weight conversion in-place on the actual weight column.
    Nulls out weights that couldn't be converted (still invalid after all attempts).
    """
    df = df.copy()

    has_weight = df["weight"].notna()
    known_type = df["product_type"].isin(WEIGHT_RANGES.keys())
    mask = has_weight & known_type

    # Convert
    df.loc[mask, "weight"] = df[mask].apply(
        lambda row: convert_weight_to_grams(row["weight"], row["product_type"]), axis=1
    )

    # Null out anything still outside valid range after conversion
    still_invalid = mask & df.apply(
        lambda row: not check_range(row["weight"], row["product_type"])
        if row["product_type"] in WEIGHT_RANGES else False,
        axis=1
    )

    nulled = still_invalid.sum()
    df.loc[still_invalid, "weight"] = None

    print(f"✅ Weight normalization applied to {mask.sum()} rows")
    print(f"🚫 Nulled out {nulled} rows that could not be converted")

    return df

# Apply to all datasets
brand_adnd_weight_normalized_datasets = [
    apply_weight_normalization(df) for df in brand_normalized_datasets_list
]

# Verify
for i, df in enumerate(brand_adnd_weight_normalized_datasets, start=1):
    valid = df[df["weight"].notna() & df["product_type"].isin(WEIGHT_RANGES.keys())]
    print(f"\nDataset {i} — {len(valid)} rows with weight after normalization")

**we normalize storage integer values so that all are in GBs**

### Step 3: Normalizing Storage

In [ ]:
import numpy as np

def normalize_storage_with_report(df, dataset_name="Dataset"):
    # 1. Store original values for comparison
    original_series = df['storage_size'].copy()
    
    # 2. Define the normalization logic
    def convert_units(row):
        size = row['storage_size']
        p_type = str(row['product_type']).upper().strip() # Added strip() for safety
        
        if pd.isna(size) or size <= 0:
            return size
            
        # Logic: Multiply by 1000 if it's not a USB and below 64 (TB threshold)
        if p_type != 'USB_STICK' and size < 64:
            return size * 1000
        return size

    # 3. Apply the transformation
    df['storage_size'] = df.apply(convert_units, axis=1)
    
    # 4. Identify changed rows
    changed_mask = ~np.isclose(original_series.fillna(-1), df['storage_size'].fillna(-1))
    df_changed = df[changed_mask]
    
    # 5. Print the Report
    print(f"\n📊 Normalization Report for {dataset_name}:")
    print(f"   - Total rows: {len(df)}")
    print(f"   - Rows changed: {len(df_changed)}")
    
    if len(df_changed) > 0:
        print("   - Sample of changes:")
        comparison = pd.DataFrame({
            'Original': original_series[changed_mask],
            'Normalized': df['storage_size'][changed_mask],
            'Product_Type': df['product_type'][changed_mask]
        }).head(5)
        print(comparison)
    
    return df

# Initialize new list
brand_storage_normalized_datasets = []

for i, df in enumerate(brand_adnd_weight_normalized_datasets, start=1):
    norm_df = normalize_storage_with_report(df.copy(), dataset_name=f"products_{i}")
    brand_storage_normalized_datasets.append(norm_df)

print(f"\n✅ Created 'brand_storage_normalized_datasets' list with {len(brand_storage_normalized_datasets)} DataFrames.")



In [ ]:
for i, df in enumerate(brand_storage_normalized_datasets, start=1):
    print(f"\n✅ Audit for Dataset {i}:")
    
    # Check if we still have any "TB-looking" numbers in HDD/SSD
    tb_range = df[(df['product_type'].isin(['HDD', 'SSD'])) & (df['storage_size'] < 64)]
    
    if tb_range.empty:
        print("  - Success: No HDD/SSDs left in the 1GB-64GB 'TB-limbo' range.")
    else:
        print(f"  - Warning: {len(tb_range)} rows still have small storage values.")
        display(tb_range[['brand', 'storage_size', 'title']].head(5))

    # Show the most common sizes to ensure they look like GB
    print("  - Top 5 Storage Capacities (GB):")
    print(df[df['product_type'].isin(['HDD', 'SSD'])]['storage_size'].value_counts().head(5))

In [ ]:
for i, df in enumerate(brand_storage_normalized_datasets):
    # Create a mask where the new value is different from the original
    # We use np.isclose for floats to handle tiny precision differences
    changed_mask = ~np.isclose(df_after_dropping[i]['storage_size'].fillna(-1), 
                               df['storage_size'].fillna(-1))
    
    # Extract the changed rows
    df_changes = df[changed_mask].copy()
    
    if not df_changes.empty:
        # Add the original value back in just for viewing
        df_changes['original_size'] = df_after_dropping[i].loc[changed_mask, 'storage_size']
        
        print(f"\n👀 Displaying {len(df_changes)} changed rows for Dataset {i+1}:")
        # Displaying key columns to verify the logic
        display(df_changes[['product_type', 'brand', 'original_size', 'storage_size']].head(10))
    else:
        print(f"\nDataset {i+1}: No changes detected.")

In [ ]:
limbo_rows = []

for i, df in enumerate(brand_storage_normalized_datasets): #all_datasets_normalized   df_after_dropping
    # Filter for the specific range and product types mentioned
    mask = (
        (df['product_type'].isin(['HDD', 'SSD'])) & 
        (df['storage_size'] > 8) & 
        (df['storage_size'] < 64)
    )
    
    df_subset = df[mask].copy()
    if not df_subset.empty:
        df_subset['dataset_source'] = f"Dataset {i+1}"
        limbo_rows.append(df_subset)

# Combine them for a single view
if limbo_rows:
    df_limbo = pd.concat(limbo_rows)
    # View the titles and brands to see what these actually are
    print(f"🔍 Found {len(df_limbo)} rows in the 16GB-128GB range for HDDs/SSDs")
    display(df_limbo[['dataset_source', 'brand', 'product_type', 'storage_size', 'title']].sort_values('storage_size'))
else:
    print("No rows found in that specific range.")

In [ ]:
#check how many hdd and ssds are above 16 and under 120 in the non-normalized datasets
for i, df in enumerate(brand_storage_normalized_datasets, start=1):
    mask = (
        (df['product_type'].isin(['HDD', 'SSD'])) & 
        # (df['storage_size'] > 16) & 
        (df['storage_size'] < 119)
    )
    count = mask.sum()
    print(f"Dataset {i+1}: {count} rows with HDD/SSD storage above 16 GB and below 119 GB before normalization.")
    #print all the rows that are in this range to check if they are actually HDDs or SSDs
    # df_subset = df[mask]
    # display(df_subset)

In [ ]:
#we check all the usb size unique values are
for i, df in enumerate(brand_storage_normalized_datasets, start=1):
    mask = (df['product_type'] == 'USB_STICK') & (df['storage_size'].notna())
    unique_sizes = df.loc[mask, 'storage_size'].unique()
    print(f"Dataset {i+1} - Unique USB Storage Sizes: {sorted(unique_sizes)}")

**we check dimensions and power metrics next

In [ ]:
attributes_to_normalize = [
    'height', 'width', 'length' ,'write_speed', 'read_speed'
]

print("=" * 60)

for attr in attributes_to_normalize:
    print(f"\n📍 ATTRIBUTE: {attr}")
    
    for i, df in enumerate(brand_storage_normalized_datasets, start=1):
        if attr not in df.columns:
            print(f"  Dataset {i}: '{attr}' column missing.")
            continue
            
        col_data = df[attr].dropna()
        if col_data.empty:
            print(f"  Dataset {i}: No data available for GPUs.")
            continue
            
        # 2. Get all unique values (sorted for readability)
        unique_vals = sorted(col_data.unique().tolist(), key=lambda x: str(x))
        
        # 3. Report
        is_numeric = pd.api.types.is_numeric_dtype(col_data)
        type_label = "Numeric" if is_numeric else "String"
        
        print(f"  Dataset {i} [{type_label}] | Unique Count: {len(unique_vals)}")
        print(f"    Options: {unique_vals}")

In [ ]:
# Grouping by general category for cleaner reporting
product_groups = {
    'GPU': ['GPU'],
    'STORAGE': ['SSD', 'HDD', 'USB_STICK']
}

attributes_to_check = [
    'height', 'width', 'length', 'write_speed', 'read_speed'
]

print("🕵️ Isolated Product-Type Audit")
print("=" * 60)

for group_name, types in product_groups.items():
    print(f"\n🚀 CATEGORY: {group_name} ({', '.join(types)})")
    print("-" * 40)
    
    for attr in attributes_to_check:
        print(f"\n📍 Attribute: {attr}")
        
        for i, df in enumerate(brand_storage_normalized_datasets, start=1):
            if attr not in df.columns:
                continue
                
            # Filter for this specific group
            mask = df['product_type'].str.upper().str.strip().isin(types)
            data = df.loc[mask, attr].dropna()
            
            if data.empty:
                continue
                
            unique_vals = sorted(data.unique().tolist())
            
            # Identify scale based on data distribution
            data_min, data_max = data.min(), data.max()
            
            print(f"  Dataset {i} | Range: [{data_min}, {data_max}] | Count: {len(unique_vals)}")
            # Show top 5 and bottom 5 if list is long
            if len(unique_vals) > 10:
                print(f"    Values: {unique_vals[:5]} ... {unique_vals[-5:]}")
            else:
                print(f"    Values: {unique_vals}")

**closer inspection as some values dont make sense**

In [ ]:
def pull_manual_checks(brand_storage_gpu_normalized):
    for i, df in enumerate(brand_storage_gpu_normalized, start=1):
        print(f"\n{'='*30} DATASET {i} SUSPECT RECORDS {'='*30}")
        
        # 1. GPU Dimension Suspects (Likely CM instead of MM)
        # Standard GPUs are > 100mm long and > 30mm thick. 
        # Anything between 1 and 60 is highly suspicious.
        gpu_mask = df['product_type'].str.upper().str.strip() == 'GPU'
        suspect_gpu_dims = df[gpu_mask & (
            (df['height'].between(0.1, 60)) | 
            (df['length'].between(0.1, 60))
        )]
        
        # 2. Storage Speed Suspects (Likely GB/s instead of MB/s)
        storage_mask = df['product_type'].str.upper().str.strip().isin(['SSD', 'HDD', 'USB_STICK'])
        suspect_speeds = df[storage_mask & (
            (df['read_speed'].between(0.1, 25)) | 
            (df['write_speed'].between(0.1, 25))
        )]
        

        # --- Display Results ---
        if not suspect_gpu_dims.empty:
            print(f"\n🚩 GPU Dims in 'CM range' ({len(suspect_gpu_dims)} rows):")
            display(suspect_gpu_dims[['brand', 'product_type', 'height', 'length', 'title']].head(10))

        if not suspect_speeds.empty:
            print(f"\n🚩 Storage Speeds in 'GB/s range' ({len(suspect_speeds)} rows):")
            display(suspect_speeds[['brand', 'product_type', 'read_speed', 'write_speed', 'title']].head(10))

# Running on current normalized list
pull_manual_checks(brand_storage_normalized_datasets)

**check unique width height and length per product type**

In [ ]:
#check unique width height and length per product type (GPU, HDD, SSD, USB_STICK) 
for i, df in enumerate(brand_storage_normalized_datasets, start=1):
    print(f"\n📍 Dataset {i} - Unique Dimensions per Product Type")
    print("=" * 80)
    for p_type in df['product_type'].unique():
        print(f"\n📍 Dataset {i} - Unique Dimensions for {p_type}")
        subset = df[df['product_type'] == p_type]
        for dim in ['height', 'width', 'length']:
            if dim in subset.columns:
                unique_vals = sorted(subset[dim].dropna().unique())
                print(f"   - {p_type} | {dim}: {unique_vals}")

### Step 4: Normalizing Dimensions

In [ ]:
import pandas as pd

# ── Audit log ────────────────────────────────────────────────────────────────
total_audit_log = {
    'GPU_cm_to_mm': 0,
    'GPU_cm_to_mm_post': 0,
    'Storage_inch_to_mm': 0,
    'Storage_cm_to_mm': 0,
    'SSD_subtype_converted': 0,
    'Untouched': 0,
    'GPU_swapped': 0,
    'GPU_bad_height_nulled': 0,
    'SSD_bad_height_nulled': 0,
}

# ── SSD subtype detection ─────────────────────────────────────────────────────
def get_ssd_subtype(row):
    form      = str(row.get('form_factor', '')   or '').upper()
    interface = str(row.get('interface_type', '') or '').upper()
    title     = str(row.get('title', '')          or '').upper()

    if any(x in form or x in title for x in ['M.2', 'M2', 'NGFF', '2280', '2242', '2260']):
        return 'M2'
    if any(x in form or x in title for x in ['2.5', '2.5"', 'SATA']):
        return '2.5'
    if 'NVME' in interface or 'NVME' in title:
        return 'M2'
    return 'UNKNOWN'

# ── SSD dimension ranges (mm) by subtype ─────────────────────────────────────
SSD_DIM_RANGES = {
    'M2': {
        'height': (1.5, 4.0),
        'width':  (20.0, 25.0),
        'length': (40.0, 110.0),
    },
    '2.5': {
        'height': (5.0, 15.0),
        'width':  (60.0, 75.0),
        'length': (90.0, 105.0),
    },
    'UNKNOWN': {
        'height': (1.5, 15.0),
        'width':  (20.0, 75.0),
        'length': (40.0, 110.0),
    }
}

def safe_unit_correction_ssd(val, dim_type, ssd_subtype):
    if pd.isna(val) or val <= 0:
        return val

    low, high = SSD_DIM_RANGES[ssd_subtype][dim_type]

    if low <= val <= high:
        return round(val, 1)

    converted_inch = round(val * 25.4, 1)
    if low <= converted_inch <= high:
        total_audit_log['Storage_inch_to_mm'] += 1
        return converted_inch

    converted_cm = round(val * 10, 1)
    if low <= converted_cm <= high:
        total_audit_log['Storage_cm_to_mm'] += 1
        return converted_cm

    # Return as-is, post-processing will decide
    return round(val, 1)

# ── Main correction function ──────────────────────────────────────────────────
def safe_unit_correction(val, row, dim_type):
    p_type = str(row.get('product_type', '') or '').upper().strip()

    if pd.isna(val) or val <= 0:
        return val

    # --- GPUs ---
    if p_type == 'GPU':
        if dim_type == 'length' and val < 60:
            total_audit_log['GPU_cm_to_mm'] += 1
            return round(val * 10, 1)
        if dim_type in ['height', 'width'] and val < 20:
            total_audit_log['GPU_cm_to_mm'] += 1
            return round(val * 10, 1)

    # --- SSDs (subtype-aware) ---
    elif p_type == 'SSD':
        ssd_subtype = get_ssd_subtype(row)
        total_audit_log['SSD_subtype_converted'] += 1
        return safe_unit_correction_ssd(val, dim_type, ssd_subtype)

    # --- HDD / USB_STICK ---
    elif p_type in ['HDD', 'USB_STICK']:
        if val < 5.0:
            total_audit_log['Storage_inch_to_mm'] += 1
            return round(val * 25.4, 1)
        if 5.0 <= val < 15.0:
            total_audit_log['Storage_cm_to_mm'] += 1
            return round(val * 10, 1)

    total_audit_log['Untouched'] += 1
    return round(val, 1)

# ── Apply to all datasets ─────────────────────────────────────────────────────
br_str_other_normalized = []

for i, df in enumerate(brand_storage_normalized_datasets, start=1):
    df_temp = df.copy()

    # Step 1: unit correction
    for col in ['height', 'width', 'length']:
        if col in df_temp.columns:
            df_temp[col] = df_temp.apply(
                lambda row, c=col: safe_unit_correction(row[c], row, c),
                axis=1
            )

    gpu_mask = df_temp['product_type'] == 'GPU'
    ssd_mask = df_temp['product_type'] == 'SSD'

    # Step 2: fix swapped height/width for GPUs
    swapped = (
        gpu_mask &
        df_temp['height'].notna() &
        df_temp['width'].notna() &
        (df_temp['height'] > df_temp['width'])
    )
    if swapped.sum() > 0:
        df_temp.loc[swapped, ['height', 'width']] = df_temp.loc[swapped, ['width', 'height']].values
        total_audit_log['GPU_swapped'] += swapped.sum()

    # Step 3: null GPU rows where height > 55mm and height < length (bad data)
    bad_gpu_height = (
        gpu_mask &
        df_temp['height'].notna() &
        df_temp['length'].notna() &
        (df_temp['height'] > 55) &
        (df_temp['height'] < df_temp['length'])
    )
    if bad_gpu_height.sum() > 0:
        df_temp.loc[bad_gpu_height, 'height'] = None
        total_audit_log['GPU_bad_height_nulled'] += bad_gpu_height.sum()

    # Step 4: null SSD heights clearly out of range (> 15mm)
    bad_ssd_height = ssd_mask & df_temp['height'].notna() & (df_temp['height'] > 15)
    if bad_ssd_height.sum() > 0:
        df_temp.loc[bad_ssd_height, 'height'] = None
        total_audit_log['SSD_bad_height_nulled'] += bad_ssd_height.sum()

    # Step 5: convert GPU rows still in cm (length < 100mm — too short to be mm)
    # excludes legitimate low-profile cards (length >= 100mm after step 1)
    # Step 5 (revised): convert GPU rows still in cm — detected by height < 20 after step 1
    still_cm = (
        gpu_mask &
        df_temp['height'].notna() &
        (df_temp['height'] < 20) &
        (df_temp['height'] > 5)   # exclude legitimate small values already in mm
    )
    if still_cm.sum() > 0:
        for col in ['height', 'width', 'length']:
            if col in df_temp.columns:
                df_temp.loc[still_cm & df_temp[col].notna(), col] = \
                    (df_temp.loc[still_cm & df_temp[col].notna(), col] * 10).round(1)
        total_audit_log['GPU_cm_to_mm_post'] += still_cm.sum()

    # Step 6: null PNY Quadro-style bad data (length > 400mm is physically impossible)
    bad_length = (
        gpu_mask &
        df_temp['length'].notna() &
        (df_temp['length'] > 400)
    )
    if bad_length.sum() > 0:
        df_temp.loc[bad_length, ['height', 'length']] = None
        total_audit_log['GPU_bad_height_nulled'] += bad_length.sum()

    br_str_other_normalized.append(df_temp)
    print(f"✅ Dataset {i} complete.")

# ── Summary ───────────────────────────────────────────────────────────────────
print("\n" + "="*50)
print("📊 GLOBAL NORMALIZATION AUDIT SUMMARY")
print("="*50)
print(f"🔹 GPU converted (CM -> MM):        {total_audit_log['GPU_cm_to_mm']}")
print(f"🔹 GPU converted post-process:      {total_audit_log['GPU_cm_to_mm_post']}")
print(f"🔹 GPU height/width swapped:        {total_audit_log['GPU_swapped']}")
print(f"🔹 GPU bad height nulled:           {total_audit_log['GPU_bad_height_nulled']}")
print(f"🔹 SSD bad height nulled:           {total_audit_log['SSD_bad_height_nulled']}")
print(f"🔹 Storage (Inches -> MM):          {total_audit_log['Storage_inch_to_mm']}")
print(f"🔹 Storage (CM -> MM):              {total_audit_log['Storage_cm_to_mm']}")
print(f"🔹 SSD subtype-aware processed:     {total_audit_log['SSD_subtype_converted']}")
print(f"🔹 Untouched (already MM):          {total_audit_log['Untouched']}")
print("="*50)

In [ ]:
for i in range(len(br_str_other_normalized)):
    df = br_str_other_normalized[i]
    print(f"\n📂 DATASET {i+1} VERIFICATION")
    
    gpu_h = df[df['product_type'] == 'GPU']['height'].dropna().unique()
    ssd_h = df[df['product_type'] == 'SSD']['height'].dropna().unique()
    
    print(f"   📍 GPU Height (mm)  : {sorted(gpu_h)}")
    print(f"   📍 SSD Height (mm)  : {sorted(ssd_h)}")


print("🔍 GPU HEIGHT OUTLIER INSPECTION")
print("=" * 80)

for i, df in enumerate(br_str_other_normalized, start=1):
    outliers = df[
        (df['product_type'] == 'GPU') & 
        (df['height'].notna()) & 
        ((df['height'] > 55) | (df['height'] < 20))
    ][['brand', 'title', 'height', 'width', 'length']].copy()
    
    if len(outliers) > 0:
        print(f"\n📂 Dataset {i} — {len(outliers)} suspicious GPU heights:")
        display(outliers)
    else:
        print(f"\n📂 Dataset {i} — ✅ No suspicious GPU heights")    

**quick check on VRAM**

In [ ]:
#check vram unique values across all datatsets br_str_gpu_other_normalized
for i, df in enumerate(br_str_other_normalized, start=1):
    if 'vram' in df.columns:
        unique_vram = sorted(df['vram'].dropna().unique())
        print(f"Dataset {i} - Unique VRAM values: {unique_vram}")
    else:
        print(f"Dataset {i} - 'vram' column missing.")

# The specific outlier values we found
outliers = [1.024, 4096.0, 6144.0, 8192.0]

print("🔍 Investigating VRAM Outlier Rows")
print("=" * 80)

for i, df in enumerate(br_str_other_normalized, start=1):
    # Filter for the specific outlier values
    outlier_df = df[df['vram'].isin(outliers)]
    
    if not outlier_df.empty:
        print(f"\n📍 Dataset {i} - Found {len(outlier_df)} rows:")
        # Displaying columns that will reveal if it's a server/workstation card
        display(outlier_df[['brand', 'title', 'vram', 'product_type']])

### Step 5: Normalizing VRAM

In [ ]:
def harmonize_vram(val):
    if pd.isna(val):
        return val
    # If the value is in the 'MB' range (e.g., 1024, 4096, 8192)
    if val >= 128:
        return round(val / 1024, 1)
    # Handle precise binary extractions like 1.024 -> 1.0
    if 1.0 < val < 1.1:
        return round(val)
    return round(val, 1)

print("⚖️ Harmonizing VRAM scales (MB to GB)...")
print("=" * 60)

for df in br_str_other_normalized:
    if 'vram' in df.columns:
        df['vram'] = df['vram'].apply(harmonize_vram)

# Verification Check
for i, df in enumerate(br_str_other_normalized, start=1):
    unique_vram = sorted(df['vram'].dropna().unique())
    print(f"Dataset {i} - Cleaned VRAM: {unique_vram}")

### Step 6: Normalizing chipset_name

In [ ]:
#check chipset name unique values across all datasets in br_str_other_normalized
for i, df in enumerate(br_str_other_normalized, start=1):
    if 'chipset_name' in df.columns:
        unique_chipsets = sorted(df['chipset_name'].dropna().unique())
        print(f"Dataset {i} - Unique Chipset Names: {unique_chipsets}")
    else:
        print(f"Dataset {i} - 'chipset_name' column missing.")

In [ ]:
vague = ['GeForce GT', 'GeForce GTX', 'Radeon', 'Radeon RX']

print("🔍 VAGUE CHIPSET INSPECTION")
print("=" * 80)

for i, df in enumerate(br_str_other_normalized, start=1):
    if 'chipset_name' not in df.columns:
        continue
    mask = df['chipset_name'].isin(vague)
    if mask.sum() == 0:
        continue
    print(f"\n📂 Dataset {i} — {mask.sum()} rows:")
    display(df[mask][['brand', 'title', 'chipset_name']].drop_duplicates())

In [ ]:
CHIPSET_NAME_MAP = {
    # ── GeForce GT ────────────────────────────────────────────────────────────
    'GeForce GT':                       None,   # too vague. couldnt find an answer
    'GeForce 210':                      'GeForce GT 210',
    'GeForce GT 210':                   'GeForce GT 210',
    'GT 1030':                          'GeForce GT 1030',
    'GeForce GT 1030':                  'GeForce GT 1030',
    'GeForce® GT 1030':                 'GeForce GT 1030',
    'NVIDIA GeForce GT 1030':           'GeForce GT 1030',
    'GeForce GT 710':                   'GeForce GT 710',
    'GeForce GT710':                    'GeForce GT 710',
    'GF GT 710':                        'GeForce GT 710',
    'Geforce GT 710':                   'GeForce GT 710',
    'Geforce GT710':                    'GeForce GT 710',
    'NVIDIA GeForce GT 710':            'GeForce GT 710',
    'NVIDIA GeForce® GT 710':           'GeForce GT 710',
    'GeForce® GT 710':                  'GeForce GT 710',
    'GeForce GT 730':                   'GeForce GT 730',

    # ── GeForce GTX 1050 ─────────────────────────────────────────────────────
    'GeForce GTX 1050':                 'GeForce GTX 1050',
    'GeForce® GTX 1050':                'GeForce GTX 1050',
    'Geforce Gtx 1050 Ti':              'GeForce GTX 1050 Ti',
    'GeForce GTX 1050 Ti':              'GeForce GTX 1050 Ti',
    'GeForce® GTX 1050 Ti':             'GeForce GTX 1050 Ti',
    'NVIDIA GeForce GTX 1050 TI':       'GeForce GTX 1050 Ti',
    'GeForce GTX 1050Ti':               'GeForce GTX 1050 Ti',
    'GEFORCE GTX 1050TI':               'GeForce GTX 1050 Ti',

    # ── GeForce GTX 1650 ─────────────────────────────────────────────────────
    'GeForce GTX':                      'GeForce GTX 1660',   # # checked manually
    'GTX 1650':                         'GeForce GTX 1650',
    'GeForce GTX 1650':                 'GeForce GTX 1650',
    'Geforce GTX 1650':                 'GeForce GTX 1650',
    'Geforce GTX 1650 Super':           'GeForce GTX 1650 Super',
    'GeForce GTX1650':                  'GeForce GTX 1650',
    'NVIDIA® GeForce GTX 1650':         'GeForce GTX 1650',
    'GTX 1650 SUPER':                   'GeForce GTX 1650 Super',
    'GTX 1650 Super':                   'GeForce GTX 1650 Super',
    'GeForce GTX 1650 SUPER':           'GeForce GTX 1650 Super',
    'GeForce GTX 1650 Super':           'GeForce GTX 1650 Super',
    'GeForce® GTX 1650 SUPER™':         'GeForce GTX 1650 Super',
    'GF GTX 1650 SUPER':                'GeForce GTX 1650 Super',
    'Geforce GTX 1650 SUPER':           'GeForce GTX 1650 Super',
    'Gtx 1650S':                        'GeForce GTX 1650 Super',
    'GEFORCE GTX 1660':                 'GeForce GTX 1660',

    # ── GeForce GTX 1660 ─────────────────────────────────────────────────────
    'GTX 1660':                         'GeForce GTX 1660',
    'GeForce GTX 1660':                 'GeForce GTX 1660',
    'GeForce GTX1660':                  'GeForce GTX 1660',
    'GeForce® GTX 1660':                'GeForce GTX 1660',
    'GEFORCE GTX 1660':                 'GeForce GTX 1660',
    'NVIDIA GeForce GTX 1660':          'GeForce GTX 1660',
    'GeForce GTX 1660 SUPER':           'GeForce GTX 1660 Super',
    'GeForce GTX 1660 Super':           'GeForce GTX 1660 Super',
    'GeForce GTX 1660 SUPER™':          'GeForce GTX 1660 Super',
    'GeForce® GTX 1660 SUPER™':         'GeForce GTX 1660 Super',
    'Geforce GTX 1660 SUPER':           'GeForce GTX 1660 Super',
    'Geforce GTX 1660 Super':           'GeForce GTX 1660 Super',
    'Geforce GTX1660 Super':            'GeForce GTX 1660 Super',
    'GeForce GTX1660S':                 'GeForce GTX 1660 Super',
    'NVIDIA GeForce GTX 1660 SUPER':    'GeForce GTX 1660 Super',
    'Geforce GTX1660 Super':            'GeForce GTX 1660 Super',
    'GTX 1660 TI':                      'GeForce GTX 1660 Ti',
    'GTX 1660 Ti':                      'GeForce GTX 1660 Ti',
    'GeForce GTX 1660 Ti':              'GeForce GTX 1660 Ti',
    'GeForce GTX 1660 TI':              'GeForce GTX 1660 Ti',
    'Geforce GTX 1660Ti':               'GeForce GTX 1660 Ti',
    'Geforce GTX 1660TI':               'GeForce GTX 1660 Ti',
    'Geforce Gtx 1660 Ti':              'GeForce GTX 1660 Ti',
    'NVIDIA® GeForce® GTX 1660 Ti':     'GeForce GTX 1660 Ti',
    'NVIDIA GeForce GTX 1660 TI':       'GeForce GTX 1660 Ti',  # if present
    'GeForce GTX 1660Ti':               'GeForce GTX 1660 Ti',

    # ── GeForce RTX 2060 ─────────────────────────────────────────────────────
    'GeForce RTX 2060':                 'GeForce RTX 2060',
    'GeForce RTX™ 2060':                'GeForce RTX 2060',
    'Geforce RTX 2060':                 'GeForce RTX 2060',
    'NVIDIA GeForce RTX 2060':          'GeForce RTX 2060',
    'Nvidia GeForce RTX 2060':          'GeForce RTX 2060',
    'RTX2060':                          'GeForce RTX 2060',
    'GeForce RTX 2060 SUPER':           'GeForce RTX 2060 Super',
    'GeForce RTX 2060 Super':           'GeForce RTX 2060 Super',
    'GeForce® RTX 2060 SUPER™':         'GeForce RTX 2060 Super',
    'RTX 2060 SUPER':                   'GeForce RTX 2060 Super',
    'RTX 2060 Super':                   'GeForce RTX 2060 Super',
    'Geforce RTX2060 Super':            'GeForce RTX 2060 Super',

    # ── GeForce RTX 2070 ─────────────────────────────────────────────────────
    'GeForce RTX 2070':                 'GeForce RTX 2070',
    'GeForce RTX™ 2070':                'GeForce RTX 2070',
    'NVIDIA GeForce RTX 2070':          'GeForce RTX 2070',
    'Geforce Rtx 2070':                 'GeForce RTX 2070',
    'RTX 2070':                         'GeForce RTX 2070',
    'GEFORCE RTX 2070 SUPER':           'GeForce RTX 2070 Super',
    'GeForce RTX 2070 SUPER':           'GeForce RTX 2070 Super',
    'GeForce RTX 2070 Super':           'GeForce RTX 2070 Super',
    'GeForce® RTX 2070 SUPER™':         'GeForce RTX 2070 Super',
    'RTX 2070 SUPER':                   'GeForce RTX 2070 Super',
    'RTX 2070 Super':                   'GeForce RTX 2070 Super',
    'RTX 2070 Super':                   'GeForce RTX 2070 Super',

    # ── GeForce RTX 2080 ─────────────────────────────────────────────────────
    'GeForce RTX 2080':                 'GeForce RTX 2080',
    'GeForce RTX 2080 SUPER':           'GeForce RTX 2080 Super',
    'GeForce RTX 2080 Super':           'GeForce RTX 2080 Super',
    'GeForce® RTX 2080 SUPER™':         'GeForce RTX 2080 Super',
    'Geforce RTX 2080 SUPER':           'GeForce RTX 2080 Super',
    'Geforce RTX 2080 Super':           'GeForce RTX 2080 Super',
    'RTX 2080 SUPER':                   'GeForce RTX 2080 Super',
    'RTX 2080 Super':                   'GeForce RTX 2080 Super',
    'NVIDIA GeForce RTX 2080 Super':    'GeForce RTX 2080 Super',
    'Rtx 2080 Super':                   'GeForce RTX 2080 Super',
    'GeForce RTX 2080 Ti':              'GeForce RTX 2080 Ti',
    'GeForce RTX 2080 TI':              'GeForce RTX 2080 Ti', # if present
    'GeForce® RTX 2080 Ti':             'GeForce RTX 2080 Ti',
    'Geforce RTX 2080 Ti':              'GeForce RTX 2080 Ti',
    'Geforce RTX 2080 TI':              'GeForce RTX 2080 Ti',
    'Geforce Rtx 2080 Ti':              'GeForce RTX 2080 Ti',
    'GF RTX 2080 Ti':                   'GeForce RTX 2080 Ti',
    'RTX 2080 TI':                      'GeForce RTX 2080 Ti',
    'RTX 2080 Ti':                      'GeForce RTX 2080 Ti',
    'GEFORCE RTX 2080TI':               'GeForce RTX 2080 Ti',

    # ── GeForce RTX 3000 ─────────────────────────────────────────────────────
    'GeForce RTX 3070':                 'GeForce RTX 3070',
    'GeForce RTX 3080':                 'GeForce RTX 3080',
    'GEFORCE RTX 3080':                 'GeForce RTX 3080',
    'Geforce RTX 3080':                 'GeForce RTX 3080',
    'GeForce RTX 3090':                 'GeForce RTX 3090',
    'GEFORCE RTX 3090':                 'GeForce RTX 3090',
    'GeForce RTX3090':                  'GeForce RTX 3090',

    # ── Radeon RX ────────────────────────────────────────────────────────────
    'Radeon':                           'Radeon RX 5500 XT',   # checked manually
    'Radeon RX':                        'Radeon RX 570',   # too vague
    'Radeon RX 550':                    'Radeon RX 550',
    'Radeon Rx 550':                    'Radeon RX 550',
    'AMD Radeon RX 560':                'Radeon RX 560',
    'Radeon RX 570':                    'Radeon RX 570',
    'AMD RX 570':                       'Radeon RX 570',
    'AMD Radeon RX 570':                'Radeon RX 570',
    'Radeon™ RX 570':                   'Radeon RX 570',
    'Radeon RX 580':                    'Radeon RX 580',
    'AMD Radeon RX 580':                'Radeon RX 580',
    'Radeon RX580':                     'Radeon RX 580',
    'Radeon RX 590':                    'Radeon RX 590',
    'Radeon™ RX 590':                   'Radeon RX 590',
    'Radeon RX 5500':                   'Radeon RX 5500 XT',
    'Radeon RX 5500 XT':                'Radeon RX 5500 XT',
    'Radeon RX 5500XT':                 'Radeon RX 5500 XT',
    'AMD Radeon RX 5500 XT':            'Radeon RX 5500 XT',
    'Radeon™ RX 5500 XT':               'Radeon RX 5500 XT',
    'RX5500XT':                         'Radeon RX 5500 XT',
    'Radeon RX 5600 XT':                'Radeon RX 5600 XT',
    'AMD Radeon RX 5600 XT':            'Radeon RX 5600 XT',
    'AMD RADEON RX5600 XT':             'Radeon RX 5600 XT',
    'RADEON RX 5600 XT':                'Radeon RX 5600 XT',
    'RX 5600 XT':                       'Radeon RX 5600 XT',
    'Radeon RX 5700':                   'Radeon RX 5700',
    'RADEON RX 5700':                   'Radeon RX 5700',
    'Radeon Rx 5700':                   'Radeon RX 5700',
    'Radeon™ RX 5700':                  'Radeon RX 5700',
    'Radeon RX 5700 XT':                'Radeon RX 5700 XT',
    'Radeon RX 5700XT':                 'Radeon RX 5700 XT',
    'AMD Radeon RX 5700 XT':            'Radeon RX 5700 XT',
    'RADEON RX 5700 XT':                'Radeon RX 5700 XT',
    'RX 5700 XT':                       'Radeon RX 5700 XT',
    'RX 5700XT':                        'Radeon RX 5700 XT',
    'RX5700XT':                         'Radeon RX 5700 XT',
    'Radeon RX5700 XT':                 'Radeon RX 5700 XT',
    'Radeon RX Vega 56':                'Radeon RX Vega 56',
    'Radeon RX VEGA 56':                'Radeon RX Vega 56',

    # ── Radeon Pro WX ────────────────────────────────────────────────────────
    'Radeon Pro WX 2100':               'Radeon Pro WX 2100',
    'Radeon Pro Wx 2100':               'Radeon Pro WX 2100',
    'Radeon Pro WX 4100':               'Radeon Pro WX 4100',
    'Radeon Pro WX4100':                'Radeon Pro WX 4100',
    'Radeon Pro WX 5100':               'Radeon Pro WX 5100',
    'Radeon Pro WX5100':                'Radeon Pro WX 5100',
    'Radeon Pro WX 7100':               'Radeon Pro WX 7100',
    'Radeon Pro WX 9100':               'Radeon Pro WX 9100',

    # ── Quadro ───────────────────────────────────────────────────────────────
    'Quadro P400':                      'Quadro P400',
    'NVIDIA Quadro P400':               'Quadro P400',
    'Quadro P1000':                     'Quadro P1000',
    'NVIDIA Quadro P1000':              'Quadro P1000',
    'Nvidia Quadro P1000':              'Quadro P1000',
    'Quadro P5000':                     'Quadro P5000',
    'NVIDIA Quadro P5000':              'Quadro P5000',
    'Quadro P6000':                     'Quadro P6000',
    'NVIDIA Quadro P6000':              'Quadro P6000',
    'NVIDIA Quadro GV100':              'Quadro GV100',
    'Quadro GV100':                     'Quadro GV100',
    'NVIDIA QUADRO GV100':              'Quadro GV100',
    'Quadro RTX 4000':                  'Quadro RTX 4000',
    'Quadro Rtx 4000':                  'Quadro RTX 4000',

    #added later:
    'GeForce RTX™ 2080 Ti':     'GeForce RTX 2080 Ti',
    'NVIDIA GeForce GT710':     'GeForce GT 710',
    'Radeon RX 5600XT':         'Radeon RX 5600 XT',
    'Geforce RTX 2060 SUPER':   'GeForce RTX 2060 Super',
    'Geforce Rtx 2060 Super':   'GeForce RTX 2060 Super',
}

def normalize_chipset_name(name):
    if pd.isna(name):
        return name
    return CHIPSET_NAME_MAP.get(str(name).strip(), name)

# Apply to all datasets
for df in br_str_other_normalized:
    if 'chipset_name' in df.columns:
        df['chipset_name'] = df['chipset_name'].apply(normalize_chipset_name)

# Verify — print any unmapped values
print("🔍 Unmapped chipset names:")
for i, df in enumerate(br_str_other_normalized, start=1):
    unmapped = df[
        df['chipset_name'].notna() &
        ~df['chipset_name'].isin(CHIPSET_NAME_MAP.values())
    ]['chipset_name'].unique()
    if len(unmapped) > 0:
        print(f"\nDataset {i}: {sorted(unmapped)}")

### Step 7: Normalizing bus_type

In [ ]:
import pandas as pd

# ============================================================
# COMPLETE BUS TYPE NORMALIZATION MAP
# Rules:
#   - No assumptions: if no lane/speed info, map to base version only
#   - Lane info (x4, x8, x16) only kept when explicitly stated
#   - Typos, casing, spacing, punctuation all handled
# ============================================================

bus_normalization_map = {

    # ----------------------------------------------------------------
    # PCIe 4.0 — base
    # ----------------------------------------------------------------
    'pci 4.0':                          'PCIe 4.0',
    'pci 4. 0':                         'PCIe 4.0',
    'pci express 4.0':                  'PCIe 4.0',
    'pci-express 4.0':                  'PCIe 4.0',
    'pci-e 4.0':                        'PCIe 4.0',
    'pcie 4.0':                         'PCIe 4.0',
    'pcie4.0':                          'PCIe 4.0',
    'gen4':                             'PCIe 4.0',
    'gen.4 pcie':                       'PCIe 4.0',
    'pci express 4.0 x16':             'PCIe 4.0 x16',
    'pci express x16 4.0':             'PCIe 4.0 x16',
    'pci-e 4.0 x 16':                  'PCIe 4.0 x16',
    'pci-e 4.0 x16':                   'PCIe 4.0 x16',
    'pcie 4.0 x16':                    'PCIe 4.0 x16',
    'pci ex 4.0 16x':                  'PCIe 4.0 x16',
    'pci ex 4.0 16x':                  'PCIe 4.0 x16',
    'pci ex 4.0 16x':                  'PCIe 4.0 x16',
    'pci ex 4.0 16x':                  'PCIe 4.0 x16',
    'pcie 4.0 x16':                    'PCIe 4.0 x16',
    'pci express 4.0 x4':              'PCIe 4.0 x4',
    'pci-express 4.0 x4':              'PCIe 4.0 x4',
    'pcie 4.0 x4':                     'PCIe 4.0 x4',
    'pcie 4.0 (x4)':                   'PCIe 4.0 x4',
    'pcie gen4 x4':                    'PCIe 4.0 x4',
    'pcie gen4x4':                     'PCIe 4.0 x4',
    'pci-express gen 4.0 x4':          'PCIe 4.0 x4',
    'pci express nvme 4.0 x4':         'PCIe 4.0 x4',
    'pcie gen 4.0 x4':                 'PCIe 4.0 x4',
    'pcie gen4':                       'PCIe 4.0',

    # ----------------------------------------------------------------
    # PCIe 3.0 — base
    # ----------------------------------------------------------------
    'pci 3.0':                         'PCIe 3.0',
    'pci express 3.0':                 'PCIe 3.0',
    'pci-express 3.0':                 'PCIe 3.0',
    'pci-e 3.0':                       'PCIe 3.0',
    'pcie 3.0':                        'PCIe 3.0',
    'pci express3.0':                  'PCIe 3.0',
    'pci express3':                    'PCIe 3.0',
    'pci express3.o':                  'PCIe 3.0',   # typo
    'pci-e3.0':                        'PCIe 3.0',
    'pcie3.0':                         'PCIe 3.0',
    'pci-e 3.0':                       'PCIe 3.0',
    'pci express 3.3':                 'PCIe 3.0',   # likely typo for 3.0
    'gen3 pcie':                       'PCIe 3.0',
    'pci-e gen3':                      'PCIe 3.0',
    'pcie gen3':                       'PCIe 3.0',
    'pcie gen 3':                      'PCIe 3.0',
    'pcie gen 3.0':                    'PCIe 3.0',   # no lane info → base
    'pci-e gen 3x4':                   'PCIe 3.0 x4',
    'pci express 3.0 x16':            'PCIe 3.0 x16',
    'pci-e 3.0 x 16':                 'PCIe 3.0 x16',
    'pci-e 3.0 x16':                  'PCIe 3.0 x16',
    'pci-e 3.0x16':                   'PCIe 3.0 x16',
    'pci-e 3.0 16x':                  'PCIe 3.0 x16',
    'pci-e 3.0 16x':                  'PCIe 3.0 x16',
    'pcie 3.0 16x':                   'PCIe 3.0 x16',
    'pcie 3.0 x16':                   'PCIe 3.0 x16',
    'pci-ex 3.0 16x':                 'PCIe 3.0 x16',
    'pci-e x 3.0 16x':                'PCIe 3.0 x16',
    'pci express x16 3.0':            'PCIe 3.0 x16',
    'pci ex 3.0 16x':                 'PCIe 3.0 x16',
    'pci ex 3.0 16x':                 'PCIe 3.0 x16',
    'pci ex 3.0 x16':                 'PCIe 3.0 x16',
    'pcie express 3.0 x 16':          'PCIe 3.0 x16',
    'pci express x16 gen3 (uses x8)': 'PCIe 3.0 x16',  # uses x8 physically but slot is x16
    'pcie x16 3.0':                   'PCIe 3.0 x16',
    'pci express 3.0 x4':             'PCIe 3.0 x4',
    'pci-express 3.0 x4':             'PCIe 3.0 x4',
    'pcie 3.0 x4':                    'PCIe 3.0 x4',
    'pcie 3.0 (x4)':                  'PCIe 3.0 x4',
    'pcie 3.0 (x2)':                  'PCIe 3.0 x2',
    'pcie 3.0x4':                     'PCIe 3.0 x4',
    'pci-e gen 3x4':                  'PCIe 3.0 x4',
    'pcie gen 3.0 x4':                'PCIe 3.0 x4',
    'pcie gen 3.0 x 4':               'PCIe 3.0 x4',
    'pcie gen 3.0 x 4 lanes':         'PCIe 3.0 x4',
    'pcie gen3 x4':                   'PCIe 3.0 x4',
    'pcie gen3 x 4':                  'PCIe 3.0 x4',
    'pcie gen3x4':                    'PCIe 3.0 x4',
    'pcie gen3. x4':                  'PCIe 3.0 x4',
    'pcie gen3 ×4':                   'PCIe 3.0 x4',
    'pcie gen3.0x 4':                 'PCIe 3.0 x4',
    'pcie gen3 8gb/s, up to 4 lanes': 'PCIe 3.0 x4',
    'pcie gen3 8 gb/s, up to 4 lanes':'PCIe 3.0 x4',
    'pcie gen3 8gb/s':                'PCIe 3.0 x4',
    'pcie gen 3 x 4':                 'PCIe 3.0 x4',
    'pcie gen 3x4':                   'PCIe 3.0 x4',
    'pcie g3 1x4':                    'PCIe 3.0 x4',
    'pcie nvme 3.0 x4':               'PCIe 3.0 x4',
    'pcie nvme gen3x4':               'PCIe 3.0 x4',
    'nvme pcie':                      'PCIe 3.0',    # no version → base (NVMe implies PCIe but not version)
    'pci express nvme 3.0 x4':        'PCIe 3.0 x4',
    'pcie gen3x2':                    'PCIe 3.0 x2',
    'pci express 3.0 x2':             'PCIe 3.0 x2',
    'pci express 3.1 x4':             'PCIe 3.1 x4',
    'pcie 3.1 x4':                    'PCIe 3.1 x4',
    'pci express x4':                 'PCIe 3.0 x4',  # only PCIe x4 → assume 3.0 is too much; keep as PCIe x4
    'pcie x4':                        'PCIe x4',
    'pci express x4':                 'PCIe x4',

    # ----------------------------------------------------------------
    # PCIe 2.0
    # ----------------------------------------------------------------
    'pci express 2.0':                'PCIe 2.0',
    'pcie 2.0':                       'PCIe 2.0',
    'pcie2.0x16':                     'PCIe 2.0 x16',
    'pcie 2.0 16x':                   'PCIe 2.0 x16',
    'pci express 2.0 x16':            'PCIe 2.0 x16',
    'pci express x16 2.0':            'PCIe 2.0 x16',
    'pci express 2.0 x16 (uses x8)':  'PCIe 2.0 x16',
    'pci express x8 2.0':             'PCIe 2.0 x8',
    'pci express 2.0 x8':             'PCIe 2.0 x8',
    'pci-e 2.0 x 8':                  'PCIe 2.0 x8',

    # ----------------------------------------------------------------
    # PCIe — no version (keep generic, no assumption)
    # ----------------------------------------------------------------
    'pcie':                           'PCIe',
    'pci express':                    'PCIe',
    'pci-express':                    'PCIe',
    'pci-e':                          'PCIe',
    'pci express x16':                'PCIe x16',
    'pci express x16':                'PCIe x16',
    'pci-express x16':                'PCIe x16',
    'pci-expressx16':                 'PCIe x16',
    'pci-ex16':                       'PCIe x16',
    'pcie x16':                       'PCIe x16',
    'pci-e x16':                      'PCIe x16',
    'pci-e 16x':                      'PCIe x16',
    'pci express x16':                'PCIe x16',
    'pci':                            'PCIe',
    'pc ex 3.0 16x':                  'PCIe 3.0 x16',  # typo of PCIe
    'pci expre':                      'PCIe',           # truncated typo
    'pci expres':                     'PCIe',           # truncated typo
    'pcie express 3.0 x 16':          'PCIe 3.0 x16',
    'pci express3.0':                 'PCIe 3.0',

    # ----------------------------------------------------------------
    # SATA 3.0 / SATA III (6 Gb/s)
    # ----------------------------------------------------------------
    'sata iii':                       'SATA III',
    'sata-iii':                       'SATA III',
    's-ata iii':                      'SATA III',
    'sataiii':                        'SATA III',
    'sata3':                          'SATA III',
    'sata3.3':                        'SATA III',
    'sata3 6gb/s':                    'SATA III',
    'sata3 6 gb/s':                   'SATA III',
    'sata3 6gbps':                    'SATA III',
    'sata3 (6gb/s)':                  'SATA III',
    'sata3/sata 6.0 gb/s':            'SATA III',
    'serial ata iii':                 'SATA III',
    'serial ata-600':                 'SATA III',
    'sata 3.0':                       'SATA III',
    'sata 3.0 (6gb/s)':               'SATA III',
    'sata 3.3 (6gb/s)':               'SATA III',
    'sata iii - 6gb/s':               'SATA III',
    'sata iii - 6.0 gb/s':            'SATA III',
    'sata iii - 6gb/s':               'SATA III',
    'sata iii 6gb/s':                 'SATA III',
    'sata iii 6 gb/s':                'SATA III',
    'sata iii 600':                   'SATA III',
    'sata iii (6 gbps)':              'SATA III',
    'sata iii (6gb/s)':               'SATA III',
    'sata iii 6gb/s':                 'SATA III',
    'sata iii 6.0gb/s':               'SATA III',
    'sata iii 6gb/s':                 'SATA III',
    'sata iii 6gb/s':                 'SATA III',
    'sata iii 6gb/s':                 'SATA III',
    'sata iii 6gb/s':                 'SATA III',
    'sata iii 6gb/s':                 'SATA III',
    'sata iii 6gb/s':                 'SATA III',
    'sata iii 6gb/s':                 'SATA III',
    'sata iii 6gb/s':                 'SATA III',
    'sata iii 6gb/s':                 'SATA III',
    'sata iii (6gb/s)':               'SATA III',
    'sata-600':                       'SATA III',
    'sata-6gbps':                     'SATA III',
    'sata-6gb/s':                     'SATA III',
    'sata/600':                       'SATA III',
    'sata600':                        'SATA III',
    'sata6gbs':                       'SATA III',
    'sata6.0gb/s':                    'SATA III',
    'sata 6 gb/s':                    'SATA III',
    'sata 6gb/s':                     'SATA III',
    'sata 6.0gb/s':                   'SATA III',
    'sata 6.0gb/s':                   'SATA III',
    'sata 6.0 gb/s':                  'SATA III',
    'sata 6.0 gbit/s':                'SATA III',
    'sata 6gbps':                     'SATA III',
    'sata 6gbs':                      'SATA III',
    'sata 6gbps':                     'SATA III',
    'sata 6gb/s':                     'SATA III',
    'sata 6g':                        'SATA III',
    'sata 6gb/sec':                   'SATA III',
    'sata 6 gbit/s':                  'SATA III',
    'sata 6gb/s':                     'SATA III',
    'sata 6gb/s':                     'SATA III',
    'sata 6gbs':                      'SATA III',
    'sata 6gb/s':                     'SATA III',
    'sata (6 gb/s)':                  'SATA III',
    'sata (sata/600)':                'SATA III',
    'sata rev 3.0':                   'SATA III',
    '6g sata':                        'SATA III',
    'sata gen 3':                     'SATA III',
    'sata 8 gb/s':                    'SATA III',       # likely data error, closest is SATA III
    'sata iii 6gb/s':                 'SATA III',
    'sata iii - 6gbps':               'SATA III',
    'serial ata iii':                 'SATA III',
    '6 gbit/s':                       'SATA III',
    '6 gbps':                         'SATA III',

    # ----------------------------------------------------------------
    # SATA II (3 Gb/s)
    # ----------------------------------------------------------------
    'sata ii':                        'SATA II',
    'sata 2':                         'SATA II',
    's-ata ii':                       'SATA II',
    'sata 3gb/s':                     'SATA II',
    'sata 3.0gb/s':                   'SATA II',
    'sata 3gb/s':                     'SATA II',
    '3g sata':                        'SATA II',
    '3g':                             'SATA II',

    # ----------------------------------------------------------------
    # SATA — no version (keep generic)
    # ----------------------------------------------------------------
    'sata':                           'SATA',
    'serial ata':                     'SATA',
    'serial ata 3.0 (sata)':          'SATA III',
    'sata 3.0gb/s':                   'SATA II',
    'sata 1':                         'SATA',           # SATA 1 = 1.5Gb/s, keep generic
    '6 gb/s':                         'SATA III',
    '6gb/s':                          'SATA III',
    '6g':                             'SATA III',

    # ----------------------------------------------------------------
    # SAS 12G (12 Gb/s)
    # ----------------------------------------------------------------
    '12g sas':                        'SAS 12G',
    '12gb/s sas':                     'SAS 12G',
    'sas 12g':                        'SAS 12G',
    'sas 12gb/s':                     'SAS 12G',
    'sas 12 gb/s':                    'SAS 12G',
    'sas 12gbps':                     'SAS 12G',
    'sas 12gb/s':                     'SAS 12G',
    'sas-12gbps':                     'SAS 12G',
    '12g sas':                        'SAS 12G',
    'sas (12gb/s sas)':               'SAS 12G',
    'sas (12gb-s sas)':               'SAS 12G',
    'sas-3 serial attached scsi v3 - 12.0gbps': 'SAS 12G',

    # ----------------------------------------------------------------
    # SAS 6G (6 Gb/s)
    # ----------------------------------------------------------------
    '6g sas':                         'SAS 6G',
    'sas 6gb/s':                      'SAS 6G',
    '6gb/s sas':                      'SAS 6G',
    'sas 6g':                         'SAS 6G',
    '6gb sas':                        'SAS 6G',
    'sas6g':                          'SAS 6G',
    'sas-6gbps':                      'SAS 6G',
    'sas 6gbps':                      'SAS 6G',
    '6gb/s sas':                      'SAS 6G',
    'sas 6gb/s':                      'SAS 6G',

    # ----------------------------------------------------------------
    # SAS 3G (3 Gb/s)
    # ----------------------------------------------------------------
    'sas 3gb/s':                      'SAS 3G',
    'sas 3gbps':                      'SAS 3G',
    'sas / serial attached scsi - 3.0gbps': 'SAS 3G',

    # ----------------------------------------------------------------
    # SAS — no version
    # ----------------------------------------------------------------
    'sas':                            'SAS',
    'serial attached scsi (sas)':     'SAS',
    'sas 2.0':                        'SAS 6G',         # SAS 2.0 spec = 6G

   # ----------------------------------------------------------------
    # USB 3.2 Gen 2x2 (20 Gbps)
    # ----------------------------------------------------------------
    'usb 3.2 gen 2x2':                'USB 3.2 Gen 2x2',
    'usb 3.2 gen2x2':                 'USB 3.2 Gen 2x2',

    # ----------------------------------------------------------------
    # USB 3.2 Gen 2 / USB 3.1 Gen 2 (10 Gbps)
    # ----------------------------------------------------------------
    'usb 3.2 gen 2':                  'USB 3.2 Gen 2',
    'usb 3.2 gen2':                   'USB 3.2 Gen 2',
    'usb 3.2 gen. 2':                 'USB 3.2 Gen 2',
    'usb 3.2 gen 2 (usb 3.1)':        'USB 3.2 Gen 2',
    'usb 3.2 gen2 (10gbps)':          'USB 3.2 Gen 2',
    'usb 3.2 gen.2 (10gbps)':         'USB 3.2 Gen 2',
    'usb 3.1 gen 2':                  'USB 3.1 Gen 2',
    'usb 3.1 gen2':                   'USB 3.1 Gen 2',
    'usb3.1 gen2':                    'USB 3.1 Gen 2',

    # ----------------------------------------------------------------
    # USB 3.2 Gen 1
    # ----------------------------------------------------------------
    'usb 3.2 gen 1':                  'USB 3.2 Gen 1',
    'usb 3.2 gen1':                   'USB 3.2 Gen 1',
    'usb 3.2 gen 1 (3.1 gen 1)':      'USB 3.2 Gen 1',
    'usb 3.2 gen 1 (usb 3.0)':        'USB 3.2 Gen 1',
    'usb type-a 3.2 gen 1 (3.1 gen 1)': 'USB 3.2 Gen 1',
    'usb-c 3.2 gen 1':                'USB 3.2 Gen 1',

    # ----------------------------------------------------------------
    # USB 3.1 Gen 1
    # ----------------------------------------------------------------
    'usb 3.1 gen 1':                  'USB 3.1 Gen 1',
    'usb 3.1 gen1':                   'USB 3.1 Gen 1',
    'usb 3.1 (gen 1)':                'USB 3.1 Gen 1',
    'usb 3.1 gen 1 (usb 3.0)':        'USB 3.1 Gen 1',
    'usb 3.0 (3.1 gen 1)':            'USB 3.1 Gen 1',
    'usb type-a 3.0 (3.1 gen 1)':     'USB 3.1 Gen 1',
    '3.0 (3.1 gen 1)':                'USB 3.1 Gen 1',
    'usb 3.2 gen 1 (3.1 gen 1)':      'USB 3.1 Gen 1',  # parenthetical clarifies it's Gen 1
    'micro-usb b 3.0 (3.1 gen 1)':    'USB 3.1 Gen 1',

    # ----------------------------------------------------------------
    # USB 3.1 — no Gen suffix (ambiguous, keep as USB 3.1)
    # ----------------------------------------------------------------
    'usb 3.1':                        'USB 3.1',
    'usb3.1':                         'USB 3.1',
    'usb 3.1/3.0 g2':                 'USB 3.1',        # ambiguous, keep generic

    # ----------------------------------------------------------------
    # USB 3.0
    # ----------------------------------------------------------------
    'usb 3.0':                        'USB 3.0',
    'usb3.0':                         'USB 3.0',
    'usb3':                           'USB 3.0',
    'usb 3':                          'USB 3.0',
    'usb 3.0 / usb 2.0':              'USB 3.0',
    'usb 3.0 and 2.0':                'USB 3.0',
    'usb 3.0 & usb-c':                'USB 3.0',
    'usb 3.0/3.1':                    'USB 3.0',
    'usb 3.0 (compatible with usb 2.0)': 'USB 3.0',
    'usb flash drive 3.0':            'USB 3.0',
    'usb 3.0 and usb 3.1':            'USB 3.0',
    'usb3.0/usb2.0':                  'USB 3.0',

    # ----------------------------------------------------------------
    # USB 3.2 — no Gen suffix (ambiguous, keep as USB 3.2)
    # ----------------------------------------------------------------
    'usb 3.2':                        'USB 3.2',
    'usb3.2':                         'USB 3.2',

    # ----------------------------------------------------------------
    # USB 2.0
    # ----------------------------------------------------------------
    'usb 2.0':                        'USB 2.0',
    'usb2.0':                         'USB 2.0',
    'hi-speed usb 2.0':               'USB 2.0',
    'otg usb':                        'USB 2.0',

    # ----------------------------------------------------------------
    # USB — no version
    # ----------------------------------------------------------------
    'usb':                            'USB',
    'usb-c':                          'USB-C',
    'usb type c':                     'USB-C',
    # ----------------------------------------------------------------
    # Thunderbolt & FireWire
    # ----------------------------------------------------------------
    'thunderbolt 3, usb 3.1 gen 2':   'Thunderbolt 3',
    'thunderbolt 3':                  'Thunderbolt 3',
    'thunderbolt 2':                  'Thunderbolt 2',
    'firewire 800 + usb 3.0':         'FireWire 800',
    'firewire 800 & usb 3.0':         'FireWire 800',


    # --- Remaining unmapped values ---

    # PCIe
    'pci express 3.0 x16 (uses x8)':  'PCIe 3.0 x16',
    'pci-express x4':                  'PCIe x4',
    'pcie 2.0 (x1)':                   'PCIe 2.0 x1',
    'pcie gen.3 x4':                   'PCIe 3.0 x4',
    'pci-express 4.0x4':               'PCIe 4.0 x4',
    'pci-express 3.0 ×16':             'PCIe 3.0 x16',   # × unicode character
    'pci-express 4x':                  'PCIe x4',         # no version stated
    'pcie 3.0 x8':                     'PCIe 3.0 x8',
    'pci express 3.0 x4 (nvme)':       'PCIe 3.0 x4',

    # SATA
    'sataiii 6gb/s':                   'SATA III',
    'sata 6gb/s (sata 3.0)':           'SATA III',
    'sata iii 6.0 gbit/s':             'SATA III',
    'sata3 (6 gb/s)':                  'SATA III',
    'sata3 6.0gb/s':                   'SATA III',
    'sata 3':                          'SATA III',
    'sata 6':                          'SATA III',        # ambiguous but 6 implies 6Gb/s
    'serial ata 6 gb/s':               'SATA III',

    # SAS
    'sas, 6gb/s':                      'SAS 6G',
    'sas 3.0gb/s':                     'SAS 3G',

    # USB
    'usb 3.':                          'USB 3.0',         # truncated
    'usb 3.5':                         'USB 3.2 Gen 2',   # not a real spec, closest is 3.2 Gen 2
    'usb 3.1 gen.2':                   'USB 3.1 Gen 2',
}


def normalize_bus_type(val):
    if pd.isna(val) or str(val).strip() == '':
        return None
    
    lookup = str(val).lower().strip()
    return bus_normalization_map.get(lookup, val)


br_str_other_final_bus_normalized = []

for i, df in enumerate(br_str_other_normalized, start=1):
    df_new = df.copy()
    
    if 'bus_type' in df_new.columns:
        original = df_new['bus_type'].copy()
        df_new['bus_type'] = df_new['bus_type'].apply(normalize_bus_type)

        valid_targets = set(bus_normalization_map.values())

        # A value is truly unmapped if:
        # - it's not null
        # - it's not in the target values
        # - it was actually present in the original (not already null)
        unmapped_mask = (
            df_new['bus_type'].notna() &
            ~df_new['bus_type'].isin(valid_targets)
        )
        unmapped = df_new.loc[unmapped_mask, 'bus_type'].unique()

        total     = len(df_new)
        mapped    = df_new['bus_type'].isin(valid_targets).sum()
        nulled    = df_new['bus_type'].isna().sum()
        unchanged = unmapped_mask.sum()

        print(f"\n📊 Dataset {i}:")
        print(f"   ✅ Mapped:    {mapped} / {total}")
        print(f"   🚫 Nulled:    {nulled}  (were empty/NaN)")
        print(f"   ⚠️  Unmapped:  {unchanged} rows with unrecognised values")
        if len(unmapped) > 0:
            print(f"   👉 Values:   {sorted(unmapped)}")

    br_str_other_final_bus_normalized.append(df_new)

print("\n🚀 Normalization complete. Data stored in 'br_str_other_final_bus_normalized'.")

In [ ]:
import pandas as pd
import numpy as np

# 1. Categorical/Mapped Pairs
categorical_checks = [
    ('brand', 'brand'),
]

# 2. All Numeric/Extracted Attributes
numeric_checks = [
    'vram', 'storage_size',
    'height', 'width', 'length', 'write_speed', 'read_speed'
]

print("📊 GLOBAL AUDIT: RAW vs. FINAL (All Product Types)")
print("=" * 90)

for i in range(len(datasets)):
    df_raw = datasets[i]
    df_final = br_str_other_final_bus_normalized[i]
    
    print(f"\n📂 DATASET {i+1} - Total Rows: {len(df_final)}")
    print("=" * 50)

    # --- Section A: Mapped Transformations ---
    print("\n--- [CATEGORICAL MAPPINGS] ---")
    for raw_col, clean_col in categorical_checks:
        if raw_col in df_raw.columns and clean_col in df_final.columns:
            raw_unique = sorted([str(x) for x in df_raw[raw_col].dropna().unique()])
            clean_unique = sorted([str(x) for x in df_final[clean_col].dropna().unique()])
            
            print(f"📍 {raw_col} ⮕ {clean_col}")
            print(f"   RAW Variations ({len(raw_unique)}): {raw_unique[:5]}...")
            print(f"   CLEAN Unique ({len(clean_unique)}): {clean_unique}")

    # --- Section B: Numeric Extractions ---
    print("\n--- [NUMERIC ATTRIBUTES] ---")
    for attr in numeric_checks:
        if attr in df_final.columns:
            # Get unique values, filtering out actual Nulls/NaNs
            clean_vals = df_final[attr].dropna().unique()
            
            if len(clean_vals) > 0:
                # Sort numerically
                clean_unique = sorted(clean_vals)
                # Format output: show a slice if list is long
                display_vals = f"{clean_unique[:12]}..." if len(clean_unique) > 12 else f"{clean_unique}"
                print(f"🔢 {attr:<25} | Uniques: {len(clean_unique):<3} | Values: {display_vals}")
            else:
                print(f"🔢 {attr:<25} | Uniques: 0   | Values: EMPTY")

print("\n🚀 Global Audit complete. Compare Storage vs. VRAM to ensure no cross-contamination.")

**check price**

In [ ]:
print("🔍 PRICE INSPECTION")
print("=" * 60)
for i, df in enumerate(br_str_other_final_bus_normalized, start=1):
    if 'price' not in df.columns:
        print(f"Dataset {i} — no price column")
        continue
    sample = df['price'].dropna().head(20).tolist()
    print(f"\nDataset {i} sample: {sample}")
    
    # Check for scientific notation strings
    sci = df['price'].dropna().astype(str).str.contains('E|e', regex=True)
    print(f"Scientific notation values: {sci.sum()}")

### Step 8: Price Normalization

In [ ]:
import re
import pandas as pd

def clean_price(val):
    if val is None or (isinstance(val, float) and pd.isna(val)):
        return None
    
    s = str(val).strip()
    
    # Fix South African Rand: "R10 819,98 incl tax" → 10819.98
    rand_match = re.match(r'^R(\d+)\s+([\d,]+)', s, re.IGNORECASE)
    if rand_match:
        whole = rand_match.group(1)
        decimal_part = rand_match.group(2).replace(',', '.')
        try:
            return round(float(whole + decimal_part), 2)
        except ValueError:
            pass
    
    # Strip currency symbols and text prefixes
    s = re.sub(r'(?i)^(SAR|MX\$|N)\s*', '', s)  # prefix currencies
    s = re.sub(r'[€£$]', '', s)                 # other currency symbols
    
    # Strip text suffixes
    s = re.sub(r'(?i)(incl|excl|tax|ron|sar|kn|km)\b.*', '', s)
    s = s.strip()
    
    # Replace comma used as decimal separator (e.g. "71,74" or "6,25")
    s = re.sub(r',(\d{2})$', r'.\1', s)
    
    # Remove remaining non-numeric chars except dot and scientific notation chars
    s = re.sub(r'[^\d.Ee+\-]', '', s)
    
    if not s:
        return None
    
    try:
        price = float(s)
    except ValueError:
        return None
    
    if price <= 0:
        return None
    
    
    return round(price, 2)


# Apply to DataFrames
br_str_other_price_normalized = []
for df in br_str_other_final_bus_normalized:
    df_temp = df.copy()
    if 'price' in df_temp.columns:
        df_temp['price'] = df_temp['price'].apply(clean_price)
    br_str_other_price_normalized.append(df_temp)

print("✅ PRICE CLEANING RESULTS")
print("=" * 60)
for i, df in enumerate(br_str_other_price_normalized, start=1):
    if 'price' not in df.columns:
        continue
    nulled = df['price'].isna().sum()
    sample = df['price'].dropna().head(10).tolist()
    print(f"\nDataset {i} — nulled: {nulled}/{len(df)}")
    print(f"Sample: {sample}")

In [ ]:
print("🔍 PRICE CHANGE INSPECTION")
print("=" * 60)

for i, (df_orig, df_clean) in enumerate(zip(br_str_other_final_bus_normalized, br_str_other_price_normalized), start=1):
    if 'price' not in df_orig.columns:
        continue
    
    changed_mask = df_orig['price'].astype(str) != df_clean['price'].astype(str)
    df_changed = pd.DataFrame({
        'original': df_orig.loc[changed_mask, 'price'],
        'cleaned':  df_clean.loc[changed_mask, 'price']
    })
    
    print(f"\n📂 Dataset {i} — {changed_mask.sum()} values changed:")
    # display(df_changed)

    #write all changes to a txt file with utf-8 encoding
    # with open(f"dataset_{i}_price_changes.txt", "w", encoding="utf-8") as f:
    #     for idx, row in df_changed.iterrows():
    #         f.write(f"Original: {row['original']} -> Cleaned: {row['cleaned']}\n")

**cast vram into int**

In [ ]:
#cast vram into int for all datasets
for i, df in enumerate(br_str_other_price_normalized, start=1):
    if 'vram' in df.columns:
        df['vram'] = pd.to_numeric(df['vram'], errors='coerce').astype('Int64')

# Verification Check
for i, df in enumerate(br_str_other_price_normalized, start=1):
    unique_vram = sorted(df['vram'].dropna().unique())
    print(f"Dataset {i} - Cleaned VRAM: {unique_vram}")        

In [ ]:
print("\n🔍 READ SPEED INSPECTION")
for i, df in enumerate(br_str_other_price_normalized, start=1):
    unique_vram = sorted(df['read_speed'].dropna().unique())
    print(f"Dataset {i} - read_speed: {unique_vram}")  

print("\n🔍 WRITE SPEED INSPECTION")    
for i, df in enumerate(br_str_other_price_normalized, start=1):
    unique_vram = sorted(df['write_speed'].dropna().unique())
    print(f"Dataset {i} - write_speed: {unique_vram}")  

print("\n🔍 STORAGE SIZE INSPECTION")
for i, df in enumerate(br_str_other_price_normalized, start=1):
    unique_vram = sorted(df['storage_size'].dropna().unique())
    print(f"Dataset {i} - storage_size: {unique_vram}")    

In [ ]:
for df in br_str_other_price_normalized:
    for col in ['read_speed', 'write_speed', 'storage_size']:
        if col in df.columns:
            df[col] = df[col].astype('Int64')  # pandas nullable integer

In [ ]:
print("\n🔍 READ SPEED INSPECTION")
for i, df in enumerate(br_str_other_price_normalized, start=1):
    unique_vram = sorted(df['read_speed'].dropna().unique())
    print(f"Dataset {i} - read_speed: {unique_vram}")  

print("\n🔍 WRITE SPEED INSPECTION")    
for i, df in enumerate(br_str_other_price_normalized, start=1):
    unique_vram = sorted(df['write_speed'].dropna().unique())
    print(f"Dataset {i} - write_speed: {unique_vram}")  

print("\n🔍 STORAGE SIZE INSPECTION")
for i, df in enumerate(br_str_other_price_normalized, start=1):
    unique_vram = sorted(df['storage_size'].dropna().unique())
    print(f"Dataset {i} - storage_size: {unique_vram}")    

**after normalizations, we add the metrics to the column names**

In [ ]:
# Mapping of old column names to new names with units
column_unit_map = {
    "width": "width_mm",
    "length": "length_mm",
    "height": "height_mm",
    "weight": "weight_g",
    "vram": "vram_gb",
    "storage_size": "storage_gb",
    "read_speed": "read_speed_mb_s",
    "write_speed": "write_speed_mb_s"
}

# Apply to each DataFrame
for i, df in enumerate(br_str_other_price_normalized, start=1):
    # Rename columns if they exist in this df
    cols_to_rename = {col: column_unit_map[col] for col in df.columns if col in column_unit_map}
    df.rename(columns=cols_to_rename, inplace=True)
    

### after curating fusion data, we do a few adjustments for normalization

In [ ]:
import numpy as np
import pandas as pd
import re

# --- 1. DEFINE ALL MAPPINGS ---

interface_map = {
    "SATA": "SATA III", "Sata": "SATA III", "Sata3": "SATA III", 
    "SataIII": "SATA III", "SATA III": "SATA III", "SATA III (6 Gbps)": "SATA III", 
    "SATA III-600": "SATA III", "SATA 6GBPS": "SATA III", "SATA-6GBPS": "SATA III", 
    "SATA-III": "SATA III", "SATA3": "SATA III", "SATA3 (6Gb/s)": "SATA III", 
    "SATA3 6Gbps": "SATA III", "Serial ATA": "SATA III", "Serial ATA 3.0 (SATA)": "SATA III", 
    "Serial ATA III": "SATA III", "Serial ATA-600": "SATA III", "Série ATA III": "SATA III",
    "SAS-12GBPS": "SAS 12G", "SAS-6GBPS": "SAS 6G", 
    "USB Type-C": "USB Type-C", "USB Type C": "USB Type-C", "USB-C": "USB Type-C", 
    "Type-C": "USB Type-C", "Type C": "USB Type-C", "TYPE-C": "USB Type-C", 
    "USB Tip C": "USB Type-C", "USB 3.1 Type C": "USB Type-C",
    "USB Type-A": "USB Type-A", "USB-A": "USB Type-A", "Type-A": "USB Type-A", 
    "USB tipo A": "USB Type-A",
    "USB 3.0": "USB 3.0", "USB3": "USB 3.0", "USB3.0": "USB 3.0", 
    "USB 3.0 and 2.0": "USB 3.0",
    "USB Type-A / USB Type-C": "USB Type-A / USB Type-C",
    "USB Type-A and USB Type-C": "USB Type-A and USB Type-C", 
    "USB-A and USB-C": "USB Type-A and USB Type-C",
    "USB-A and Lightning": "USB Type-A and Lightning",
    "micro USB OTG": "USB 2.0 and OTG", "PCIe": "PCIe"
}

bus_type_map = {
    'SAS 12G': 'SAS', 'SAS 3G': 'SAS', 'SAS 6G': 'SAS',
    'SATA II': 'SATA', 'SATA III': 'SATA',
    'Thunderbolt 2': 'Thunderbolt', 'Thunderbolt 3': 'Thunderbolt',
    'USB 3.0': 'USB 3.0', 'USB 3.1': 'USB 3.0', 'USB 3.1 Gen 1': 'USB 3.0', 
    'USB 3.2': 'USB 3.0', 'USB 3.2 Gen 1': 'USB 3.0',
    'USB 3.2 Gen 2x2': 'USB 3.2 Gen 2',
    'USB-C': 'USB Type-C',
    'PCIe 3.0 x4': 'PCI Express x4', 'PCIe 3.1 x4': 'PCI Express x4', 
    'PCIe 4.0 x4': 'PCI Express x4', 'PCIe x4': 'PCI Express x4',
    'PCIe 2.0 x16': 'PCI Express x16', 'PCIe 3.0': 'PCI Express x16', 
    'PCIe 3.0 x16': 'PCI Express x16', 'PCIe 4.0 x16': 'PCI Express x16', 
    'PCIe x16': 'PCI Express x16',
    'PCIe 2.0 x8': 'PCI Express x8', 'PCIe 3.0 x8': 'PCI Express x8',
    'PCIe 3.0 x2': 'PCI Express x2'
}

ff_groups = {
    '2.5-inch': ['1"', '2,5', '2,5"', "2,5''", '2,5Zoll', '2.5', '2.5 "', '2.5 INCH', '2.5 Inch', '2.5 in × 7 mm', '2.5 inch', '2.5 inch 7mm', '2.5 inch SFF', '2.5"', '2.5" (6.4 cm)', '2.5" (in 3.5" carrier)', '2.5" - 3.5"', '2.5" / 7mm', '2.5" 7mm', '2.5" SFF', '2.5" Slim', '2.5". 7mm', '2.5"/7mm', "2.5'", "2.5''", "2.5'' SFF", '2.5-Inch', '2.5-inch', '2.5-inch (SFF)', '2.5-inch SFF', '2.5-inch internal SSD', '2.5INCH', '2.5Inch', '2.5P', '2.5\\', '2.5``', '2.5in', '2.5in 7mm', '2.5inch', '2.5inch 7mm', '2.5inx15mm SFF', '2.5s', '2.5“', '2.5”', '2.5” 7mm', '2.5″', '6.4 cm (2.5)', 'SFF', 'SFF (2.5-inch)', 'SFF (2.5in)', 'SFF 2.5"', 'SFF 2.5Inches'],
    '3.5-inch': ['2.713” H x 5.70” L, Single Slot', '3 5in', '3,5', '3,5"', "3,5''", '3. 5-lff', '3.12.7 cm (5")', '3.5', '3.5 "', '3.5 INCH', '3.5 Inch', '3.5 Inch x 1/3H', '3.5 X 1/3h', '3.5 col', '3.5 inch', '3.5 inches', '3.5"', '3.5" Form Factor', '3.5" LFF', '3.5" x 1/3H', '3.5"x1.0" (LFF Hard Drive)', "3.5'", "3.5''", '3.5-Inch', '3.5-inch', '3.5-inch LFF', '3.50 inches', '3.5IN', '3.5INCH', '3.5Inch', '3.5in', '3.5inch', '3.5”', '3.5″', 'LFF', 'LFF (3.5-inch)', 'LFF (3.5in)', 'LFF 3.5"'],
    'M.2 2280': ['M.2', 'M.2 (2280)', 'M.2 (22x80)', 'M.2 2280', 'M.2 2280 (80mm)', 'M.2 2280 double-sided', 'M.2 2280SS', 'M.2(2280)', 'M.2-2280', 'M2 2280', 'm2_2280', '22x80 mm', '22x80mm', '2280', 'NVMe-M.2', 'Carte M.2'],
    'mSATA': ['MSATA', 'mSATA'],
    'External': ['External', 'PORTABLE', 'Portable', 'Portable External', 'external SSD'],
    'Single Slot': ['Single Slot', 'Single-Slot', 'single-slot PCIe', 'single-slot PCIe form factor', 'single-slot, low-profile'],
    'Dual Slot': ['Dual Slot', 'Dual-Slot'],
    'ATX': ['ATX', 'ATX Form']
}

sc_groups = {
    "2.5-inch External": ['2.5" External'],
    "2.5-inch SAS": ['2.5 Inch SAS', '2.5 SAS', '2.5 inch SAS', '2.5" SAS', "2.5'' SAS", '2.5-inch SAS', '2.5Inches SAS', '2.5″ SAS'],
    "2.5-inch SATA": ['2,5-inch SATA', '2.5 INCH', '2.5 Inch', '2.5 Inch SATA', '2.5 SATA', '2.5 SATA3', '2.5 SFF', '2.5 inch', '2.5 inch SATA', '2.5" SATA', '2.5" SATA3', '2.5" Sata III', "2.5' SATA", "2.5'' SATA", "2.5'' SATA3", '2.5-Inch', '2.5-Inch SATA', '2.5-Inch SSD', '2.5-inch', '2.5-inch SATA', '2.5IN SATA III', '2.5P SATA III', '2.5SATA', '2.5in', '2.5in SATA', '2.5inch', '2.5inch SATA-6GBPS', '2.5” SATA', '2.5″ SATA', 'SataIII2.5'],
    "3.5-inch External": ["3.5'' External"],
    "3.5-inch SAS": ['3.5 Inch SAS', '3.5 SAS', '3.5 inch sas', '3.5" SAS', '3.5-inch SAS', '3.5-inch LFF SAS'],
    "3.5-inch SATA": ['3,5 SATA', '3,5" SATA', '3.5 - sata', '3.5 - sata 6Gb/s', '3.5 SATA', '3.5 inch SATA', '3.5" - SATA', '3.5" 6G SATA', '3.5" SATA', '3.5" SATA III', '3.5" SATA3', '3.5" Sata3', "3.5' SATA", "3.5'' SATA", "3.5'SATA3", '3.5-Inch SATA', '3.5-inch SATA', '3.5IN SATA', '3.5Inch SATA', '3.5in SATA', '3.5” SATA', 'LFF (3.5-inch) SATA', 'LFF SATA'],
    "M.2": ['M.2', 'M2'], "U2": ['U.2', 'U.2 2.5-Inch'], "USB": ['USB', 'USB3'], "USB 3.0": ['USB 3.0'],
    "USB Type-C": ['USB 3.1 Type C', 'USB Type C', 'Type C', 'Type-C', 'USB Type-C', 'USB-C'],
    "USB Type-A": ['Type-A', 'USB Type-A', 'USB-A', 'USB tipo A'],
    "USB Type-A and USB Type-C": ['USB Type-A / USB Type-C'],
    "USB Type-A and Lightning": ['USB-A and Lightning'],
    "USB Type-A and Micro-USB": ['USB Type-A and Micro-USB'],
    "mSATA": ['mSATA']
}

# Generate Flat Maps
form_factor_map = {val: key for key, values in ff_groups.items() for val in values}
storage_conn_map = {val: key for key, values in sc_groups.items() for val in values}
artifacts = ['â€¢', 'Â®', 'â€™', '@', 'â', 'Â', '™'] # Added broader artifact check
artifact_pattern = '|'.join(map(re.escape, artifacts))

# --- 2. THE LOGIC FUNCTION ---
def normalize_dimensions_logic(row):
    l, w, h = row['length_mm'], row['width_mm'], row['height_mm']
    if pd.isna(l) or pd.isna(w) or pd.isna(h):
        return l, w, h
    
    # Sort numeric values descending: [Largest, Middle, Smallest]
    s = sorted([l, w, h], reverse=True)
    
    if row.get('product_type') == 'GPU':
        # GPU Rule: Max -> Length, Mid -> Height, Min -> Width
        return s[0], s[2], s[1]
    else:
        # Others Rule: Max -> Length, Mid -> Width, Min -> Height
        return s[0], s[1], s[2]

# --- 3. THE MASTER PROCESSING LOOP ---


for i, df in enumerate(br_str_other_price_normalized, start=1):
    print(f"\n📊 DataFrame {i}")

    # A. Clean Text Columns
    for col in ['description', 'title']:
        if col in df.columns:
            df[col] = df[col].astype(str).str.replace(artifact_pattern, '', regex=True).str.strip()
        else:
            print(f"⚠️ Column '{col}' not found → text cleaning skipped")

    # B. Apply Mappings
    if 'bus_type' in df.columns:
        df['bus_type'] = df['bus_type'].replace(bus_type_map)
    if 'form_factor' in df.columns:
        df['form_factor'] = df['form_factor'].replace(form_factor_map)
    if 'storage_connection_type' in df.columns:
        df['storage_connection_type'] = df['storage_connection_type'].replace(storage_conn_map)
    
    t_int = 'interface' if 'interface' in df.columns else 'interface_type'
    if t_int in df.columns:
        df[t_int] = df[t_int].replace(interface_map)

    # C. Fix Dimensions + summary
    dim_cols = ['length_mm', 'width_mm', 'height_mm']

    if not set(dim_cols).issubset(df.columns):
        missing = set(dim_cols) - set(df.columns)
        print(f"⚠️ Skipped dimensions → missing columns: {missing}")
    else:
        for col in dim_cols:
            df[col] = pd.to_numeric(df[col], errors='coerce')

        total_rows = len(df)
        nan_rows = df[dim_cols].isna().any(axis=1).sum()
        valid_rows = total_rows - nan_rows

        df[dim_cols] = df.apply(
            lambda r: pd.Series(normalize_dimensions_logic(r), index=dim_cols),
            axis=1
        )

        print(f"✔️ Dimensions: {valid_rows}/{total_rows} rows processed ({nan_rows} skipped)")


**save the current result to output dir**

In [ ]:
import os
import json
import numpy as np
import pandas as pd
from pathlib import Path

# Paths relative to this notebook
NOTEBOOK_DIR = Path(".").resolve()
INPUT_DIR = NOTEBOOK_DIR / "input"
OUTPUT_DIR = NOTEBOOK_DIR / "output"
NORMALIZED_OUTPUT_DIR = INPUT_DIR / "data_cleaned_final"
os.makedirs(NORMALIZED_OUTPUT_DIR, exist_ok=True)

for i, df in enumerate(br_str_other_price_normalized, start=1):
    output_path = NORMALIZED_OUTPUT_DIR / f"dataset_{i}_normalized.json"
    
    # 1. Handle NaN/NaT/NA: Replace all with None so JSON outputs 'null'
    # We use .where(pd.notnull(df), None) as a safety net for nullable types
    df_clean = df.replace({np.nan: None}).where(pd.notnull(df), None)
    
    # 2. Convert to list of dictionaries
    data_dict = df_clean.to_dict(orient='records')
    
    # 3. Write to file with indentation for the "under each other" look
    with open(output_path, 'w', encoding='utf-8') as f:
        json.dump(data_dict, f, indent=4, ensure_ascii=False)
        
    print(f"✅ Dataset {i} saved as JSON to {output_path}")

In [ ]:
coverage = profiler.analyze_coverage(
    datasets=br_str_other_price_normalized,
    include_samples=True,
    sample_count=3  # Show 3 sample values per attribute
)
# 1. Force pandas to show all 66 rows
pd.set_option('display.max_rows', None)

# 2. Force pandas to show all 20 columns
pd.set_option('display.max_columns', None)

# 3. Prevent text inside cells from being cut off
pd.set_option('display.max_colwidth', None)

print("📊 Attribute coverage across datasets:")
display(coverage)

# Identify attributes suitable for entity matching
print("\n🔗 Attributes suitable for entity matching:")
matching_attrs = coverage[coverage['datasets_with_attribute'] >= 2]['attribute'].tolist()
print(f"Attributes available in 2+ datasets: {matching_attrs}")

#### End of Normalization